# 📈 Market Dynamics Analyzer — Analysis Notebook

This notebook replicates the Python backend of the **Market Dynamics Analyzer** app as a fully self-contained Jupyter environment.

Run all cells top-to-bottom:

| # | Section |
|---|---|
| 1 | Setup & Imports |
| 2 | Configuration |
| 3 | Data Loading & Preprocessing |
| 4 | Regression Utility Functions |
| 5 | Regression Fitting (Linear · Poly · Multivariate) |
| 6 | Price Bridging (optional) |
| 7 | EPAM Simulation & Forecasting |
| 8 | Visualizations |
| 9 | Summary Table |


In [137]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.integrate import odeint
from scipy.interpolate import interp1d

# ── Plot Theme: light mode (matches app theme.py) ─────────────────────
T = {
    "plot_paper": "#FFFFFF",
    "plot_bg":    "#FFFFFF",
    "grid":       "rgba(0,0,0,0.07)",
    "heading":    "#000000",
    "text_muted": "#000000",
    "text":       "#000000",
    "marker":     "#1E3A5F",
    "line_blue":  "#2563EB",
    "line_red":   "#DC2626",
    "line_green": "#059669",
    "line_pink":  "#7C3AED",
    "accent":     "#2563EB",
    "scene_bg":   "#FFFFFF",
    "scene_grid": "rgba(0,0,0,0.10)",
    "scene_zero": "rgba(0,0,0,0.15)",
    "font":       "Inter, sans-serif",
}

def get_layout(title, x, y):
    return dict(
        title=dict(text=title, font=dict(color=T['heading'], size=16)),
        xaxis=dict(title=x, title_font=dict(color=T['text_muted'], size=13),
                   tickfont=dict(color=T['text_muted']), gridcolor=T['grid'], zeroline=False),
        yaxis=dict(title=y, title_font=dict(color=T['text_muted'], size=13),
                   tickfont=dict(color=T['text_muted']), gridcolor=T['grid'], zeroline=False),
        paper_bgcolor=T['plot_paper'], plot_bgcolor=T['plot_bg'],
        legend=dict(font=dict(color=T['text']), bgcolor='rgba(0,0,0,0)'),
        margin=dict(l=20, r=20, t=50, b=20),
        font=dict(family=T['font']),
    )


import pathlib

def save_fig(fig, filename):
    """Show the figure in the notebook and export it as a hi-res PNG."""
    fig.show()
    out = pathlib.Path('Simulation_Result') / item_name / filename
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out), scale=2)
    print(f'  ✅ Saved → {out}')

print('✅ Imports and theme loaded.')


✅ Imports and theme loaded.


## ⚙️ Section 2 — Configuration

Edit the values below to match your dataset.

- **`n_fit`** — rows used to *fit* all regression models
- **`n_forecast`** — rows held out as ground truth for the EPAM forecast evaluation

> ⚠️ `n_fit + n_forecast` must not exceed the total number of rows in your CSV.


In [ ]:
CONFIG = {
    "data_path":        "./simulation_data/AMPLOP 90 PPS PPL_data.csv",
    "supply_qty_col":   "Qty Beli",
    "demand_qty_col":   "Qty Jual",
    "supply_price_col": "Harga Beli",
    "demand_price_col": "Harga Jual",
    # ── Row Split ────────────────────────────────────────────────────
    "n_fit":      48,    # rows used for regression fitting
    "n_forecast": 12,    # rows held out for EPAM forecast evaluation
    # ── Data Cleaning ────────────────────────────────────────────────
    "use_imputation":        True,
    "treat_zero_as_missing": True,
    # ── Regression ───────────────────────────────────────────────────
    "poly_degree": 3,
    # ── Price Bridging ───────────────────────────────────────────────
    "use_bridging":    False,
    "bridging_master": None,   # e.g. "Demand Price"
    "bridging_degree": 1,
    # ── EPAM ─────────────────────────────────────────────────────────
    "calc_method": "Dynamic",  # "Dynamic" or "Constant"
    "rolling_k_window": 12,   # max lookback for Rolling Window K
}

print('✅ CONFIG loaded.')

# ── Item name (used as export subfolder) ─────────────────────────────
import pathlib as _pl
item_name = _pl.Path(CONFIG['data_path']).stem.replace('_data', '').strip()
print(f'Item: {item_name}')


✅ CONFIG loaded.
Item: AMPLOP 90 PPS PPL


## 📂 Section 3 — Data Loading & Preprocessing

Steps:
1. Load CSV → `df_raw`
2. Numeric validation, zero-treatment, MICE imputation on `df_raw`
3. Validate `n_fit + n_forecast ≤ len(df_raw)`
4. Split into `df_fit` (train) and `df_forecast_true` (hold-out)


In [139]:
# ── Unpack config ─────────────────────────────────────────────────────
supply_qty_col   = CONFIG['supply_qty_col']
demand_qty_col   = CONFIG['demand_qty_col']
supply_price_col = CONFIG['supply_price_col']
demand_price_col = CONFIG['demand_price_col']
n_fit            = CONFIG['n_fit']
n_forecast       = CONFIG['n_forecast']
poly_degree      = CONFIG['poly_degree']
use_imputation   = CONFIG['use_imputation']
treat_zero       = CONFIG['treat_zero_as_missing']
use_bridging     = CONFIG['use_bridging']
bridging_master  = CONFIG['bridging_master']
bridging_degree  = CONFIG['bridging_degree']
calc_method      = CONFIG['calc_method']

# ── Load ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CONFIG['data_path'])

# ── Numeric validation ────────────────────────────────────────────────
cols_to_use = [supply_qty_col, demand_qty_col, supply_price_col, demand_price_col]
for col in cols_to_use:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# ── Zero treatment ────────────────────────────────────────────────────
if treat_zero:
    num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    df_raw[num_cols] = df_raw[num_cols].replace(0, np.nan)

# ── Split BEFORE imputation (avoids data leakage) ────────────────────
assert n_fit + n_forecast <= len(df_raw), (
    f"n_fit ({n_fit}) + n_forecast ({n_forecast}) = {n_fit+n_forecast} "
    f"exceeds dataset size ({len(df_raw)} rows)")

df_fit           = df_raw.iloc[:n_fit].copy().reset_index(drop=True)
df_forecast_true = df_raw.iloc[n_fit:n_fit+n_forecast].copy().reset_index(drop=True)

# ── MICE Imputation (fit on training data only) ──────────────────────
num_cols = df_fit.select_dtypes(include=[np.number]).columns.tolist()
if use_imputation and df_fit[cols_to_use].isnull().values.any():
    imputer = IterativeImputer(random_state=42)
    df_fit[num_cols] = imputer.fit_transform(df_fit[num_cols])
    # Transform hold-out with the SAME fitted imputer (no leakage)
    if n_forecast > 0 and df_forecast_true[cols_to_use].isnull().values.any():
        df_forecast_true[num_cols] = imputer.transform(df_forecast_true[num_cols])
    print('MICE imputation applied (fit on training data only).')

print(f'Total rows: {len(df_raw)} | Fit: {len(df_fit)} | Forecast hold-out: {len(df_forecast_true)}')

print('\nFit data (head):')
display(df_fit.head())
print('\nForecast ground truth:')
display(df_forecast_true)
rolling_k_window = CONFIG['rolling_k_window']


MICE imputation applied (fit on training data only).
Total rows: 73 | Fit: 24 | Forecast hold-out: 12

Fit data (head):


,Date,Nama,Qty Beli,Harga Beli,Qty Jual,Harga Jual
0,2019-01,AMPLOP 90 PPS PPL,1568.316877,13503.717177,1462.6875,15305.565646
1,2019-02,AMPLOP 90 PPS PPL,840.540501,13423.191634,2.0000,15274.050336
2,2019-03,AMPLOP 90 PPS PPL,1050.000000,12903.000000,1050.0000,14850.000000
3,2019-04,AMPLOP 90 PPS PPL,1568.316877,13503.717177,1462.6875,15305.565646
4,2019-05,AMPLOP 90 PPS PPL,900.000000,13167.000000,900.0000,14850.000000



Forecast ground truth:


,Date,Nama,Qty Beli,Harga Beli,Qty Jual,Harga Jual
0,2021-01,AMPLOP 90 PPS PPL,862.018687,13973.66357,222.0,15916.0
1,2021-02,AMPLOP 90 PPS PPL,3000.000000,14424.00000,1614.0,16383.0
2,2021-03,AMPLOP 90 PPS PPL,1500.000000,15183.00000,1897.0,16716.0
3,2021-04,AMPLOP 90 PPS PPL,3150.000000,15982.00000,3038.0,17383.0
4,2021-05,AMPLOP 90 PPS PPL,2400.000000,15982.00000,3154.0,18266.0
5,2021-06,AMPLOP 90 PPS PPL,1500.000000,15982.00000,1876.0,18266.0
6,2021-07,AMPLOP 90 PPS PPL,1500.000000,15982.00000,1498.0,18283.0
7,2021-08,AMPLOP 90 PPS PPL,1500.000000,15982.00000,1485.0,18283.0
8,2021-09,AMPLOP 90 PPS PPL,4500.000000,15982.00000,2902.0,18283.0
9,2021-10,AMPLOP 90 PPS PPL,1500.000000,15982.00000,2205.0,18283.0


## 🔧 Section 4 — Regression Utility Functions

Verbatim port of `regression_utils.py`. Functions:

| Function | Purpose |
|---|---|
| `perform_linear_regression` | Fit OLS linear model |
| `perform_polynomial_regression` | Fit polynomial model |
| `calculate_dynamic_k` | Compute time-varying price adjustment speed K |
| `solve_epam` | Solve EPAM ODE (autonomous) |
| `perform_multivariate_regression` | Fit Q = b0 + b1·P + b2·t |
| `solve_non_autonomous_epam` | Solve EPAM ODE (non-autonomous, time-aware) |


In [140]:
def compute_rolling_k(k_raw, t_idx, window_n):
    """Rolling-window OLS *without intercept*: K = beta * t.

    At each step i the window covers indices [max(0, i+1-window_n) .. i].
    For forecasting, call iteratively: append predicted K, slide window.
    """
    k_roll = np.zeros(len(k_raw))
    last_beta = 0.0
    for i in range(len(k_raw)):
        lo = max(0, i + 1 - window_n)
        t_w = t_idx[lo:i+1].astype(float).reshape(-1, 1)
        k_w = k_raw[lo:i+1]
        denom = float(t_w.T @ t_w)
        if abs(denom) > 1e-12:
            last_beta = float(t_w.T @ k_w) / denom
        k_roll[i] = last_beta * t_idx[i]
    return k_roll, last_beta

def forecast_rolling_k(k_hist, t_hist, t_fc, window_n):
    """Iteratively forecast K into t_fc using rolling OLS (no intercept)."""
    all_k = list(k_hist)
    all_t = list(t_hist)
    fc_k = []
    for t_new in t_fc:
        lo = max(0, len(all_k) - window_n)
        t_w = np.array(all_t[lo:], dtype=float).reshape(-1, 1)
        k_w = np.array(all_k[lo:])
        denom = float(t_w.T @ t_w)
        beta = float(t_w.T @ k_w) / denom if abs(denom) > 1e-12 else 0.0
        k_new = beta * float(t_new)
        fc_k.append(k_new)
        all_k.append(k_new)
        all_t.append(float(t_new))
    return np.array(fc_k)

def perform_loglinear_regression(X, y):
    """Fit ln(Q) = a + b*P via OLS.  Returns model, y_pred (original scale), mape."""
    y_log = np.log(np.maximum(y, 1e-10))
    model = LinearRegression()
    model.fit(X, y_log)
    y_pred = np.exp(model.predict(X))
    mape = mean_absolute_percentage_error(y, y_pred)
    return model, y_pred, mape

def perform_linear_regression(X, y):
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    mape = mean_absolute_percentage_error(y, y_pred)
    return model, y_pred, mape

def perform_polynomial_regression(X, y, degree=2):
    poly_features = PolynomialFeatures(degree=degree)
    X_poly = poly_features.fit_transform(X)
    model = LinearRegression()
    model.fit(X_poly, y)
    y_pred = model.predict(X_poly)
    mape = mean_absolute_percentage_error(y, y_pred)
    return model, poly_features, y_pred, mape

def calculate_dynamic_k(prices, q_demand_func, q_supply_func):
    k_values = []
    for i in range(len(prices) - 1):
        P_curr, P_next = prices[i], prices[i+1]
        gap = q_demand_func(P_curr) - q_supply_func(P_curr)
        delta_P = P_next - P_curr
        k_t = delta_P / gap if abs(gap) > 1e-4 else (k_values[-1] if k_values else 0.0)
        k_values.append(k_t)
    k_values.append(k_values[-1] if k_values else 0.0)
    k_arr = np.array(k_values)
    t_idx = np.arange(len(prices))
    k_func = interp1d(t_idx, k_arr, kind='previous', fill_value='extrapolate')
    return k_arr, k_func

def solve_epam(P0, k_func, q_d_func, q_s_func, t_steps):
    def ode(P, t, kf, qd, qs):
        Pv = P[0] if isinstance(P, (list, np.ndarray)) else P
        return kf(t) * (qd(Pv) - qs(Pv))
    return odeint(ode, P0, t_steps, args=(k_func, q_d_func, q_s_func)).flatten()

def perform_multivariate_regression(df, price_col, time_col, qty_col):
    X = df[[price_col, time_col]].values
    y = df[qty_col].values
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    mape = mean_absolute_percentage_error(y, y_pred)
    return model, (model.intercept_, model.coef_), mape

def solve_non_autonomous_epam(P0, k_input, d_func, s_func, t_steps):
    def ode(P, t, ki, df_, sf_):
        Pv = P[0] if isinstance(P, (list, np.ndarray)) else P
        kt = ki(t) if callable(ki) else ki
        return kt * (df_(Pv, t) - sf_(Pv, t))
    return odeint(ode, P0, t_steps, args=(k_input, d_func, s_func)).flatten()

print('✅ Regression utilities defined.')


✅ Regression utilities defined.


In [141]:
# ── Adjusted Gap utilities ────────────────────────────────────────────

def compute_adjusted_gap(q_demand_pred, q_supply_pred):
    """Compute raw gap, mean gap (structural bias), and adjusted gap."""
    gap_raw = q_demand_pred - q_supply_pred
    gap_mean = float(np.mean(gap_raw))
    gap_adj = gap_raw - gap_mean
    return gap_raw, gap_mean, gap_adj


def solve_epam_adjusted_gap(P0, k_input, gap_adj, t_steps):
    """Continuous EPAM with pre-computed adjusted gap via odeint.
    dP/dt = k(t) * gap_adj_interp(t)."""
    gap_interp = interp1d(t_steps, gap_adj, kind='previous',
                          fill_value='extrapolate')
    def ode(P, t):
        kt = k_input(t) if callable(k_input) else k_input
        return kt * gap_interp(t)
    return odeint(ode, P0, t_steps).flatten()

print('✅ Adjusted Gap utilities defined.')


✅ Adjusted Gap utilities defined.


## 📐 Section 5 — Regression Fitting

Fit all three regression models on `df_fit`.


### 5a — Linear Regression


In [142]:
time_steps_fit = np.arange(n_fit)
time_steps_fc  = np.arange(n_fit, n_fit + n_forecast)

X_demand_raw = df_fit[[demand_price_col]].values
X_supply_raw = df_fit[[supply_price_col]].values
y_demand     = df_fit[demand_qty_col].values
y_supply     = df_fit[supply_qty_col].values

lin_model_d, y_pred_lin_d, mape_lin_d = perform_linear_regression(X_demand_raw, y_demand)
lin_model_s, y_pred_lin_s, mape_lin_s = perform_linear_regression(X_supply_raw, y_supply)

print(f'Linear Demand → intercept={lin_model_d.intercept_:.4f}, coef={lin_model_d.coef_[0]:.4f}, MAPE={mape_lin_d:.2%}')
print(f'Linear Supply → intercept={lin_model_s.intercept_:.4f}, coef={lin_model_s.coef_[0]:.4f}, MAPE={mape_lin_s:.2%}')


Linear Demand → intercept=518.8750, coef=0.0617, MAPE=3057.75%
Linear Supply → intercept=6012.6712, coef=-0.3291, MAPE=26.93%


### 5b — Polynomial Regression


In [143]:
poly_model_d, poly_feat_d, y_pred_poly_d, mape_poly_d = perform_polynomial_regression(X_demand_raw, y_demand, degree=poly_degree)
poly_model_s, poly_feat_s, y_pred_poly_s, mape_poly_s = perform_polynomial_regression(X_supply_raw, y_supply, degree=poly_degree)

print(f'Poly Demand → intercept={poly_model_d.intercept_:.4f}, MAPE={mape_poly_d:.2%}')
print(f'Poly Supply → intercept={poly_model_s.intercept_:.4f}, MAPE={mape_poly_s:.2%}')


Poly Demand → intercept=-17787401.8870, MAPE=3197.13%
Poly Supply → intercept=845248.7939, MAPE=27.19%


### 5c — Multivariate Regression (Price × Time)


In [144]:
df_fit_na = df_fit.copy()
df_fit_na['Time_Index'] = time_steps_fit

model_d_na, (na_d_int, na_d_coefs), mape_d_na = perform_multivariate_regression(
    df_fit_na, demand_price_col, 'Time_Index', demand_qty_col)
model_s_na, (na_s_int, na_s_coefs), mape_s_na = perform_multivariate_regression(
    df_fit_na, supply_price_col, 'Time_Index', supply_qty_col)

print(f'MV Demand → b0={na_d_int:.4f}, b_P={na_d_coefs[0]:.4f}, b_t={na_d_coefs[1]:.4f}, MAPE={mape_d_na:.2%}')
print(f'MV Supply → b0={na_s_int:.4f}, b_P={na_s_coefs[0]:.4f}, b_t={na_s_coefs[1]:.4f}, MAPE={mape_s_na:.2%}')


MV Demand → b0=8168.1279, b_P=-0.4688, b_t=40.8647, MAPE=2203.51%
MV Supply → b0=10859.1723, b_P=-0.7118, b_t=27.8646, MAPE=28.92%


### 5d — Log-Linear Regression


In [145]:
loglin_model_d, y_pred_loglin_d, mape_loglin_d = perform_loglinear_regression(X_demand_raw, y_demand)
loglin_model_s, y_pred_loglin_s, mape_loglin_s = perform_loglinear_regression(X_supply_raw, y_supply)

print(f'Log-Linear Demand → a={loglin_model_d.intercept_:.4f}, b={loglin_model_d.coef_[0]:.6f}, MAPE={mape_loglin_d:.2%}')
print(f'Log-Linear Supply → a={loglin_model_s.intercept_:.4f}, b={loglin_model_s.coef_[0]:.6f}, MAPE={mape_loglin_s:.2%}')


Log-Linear Demand → a=6.2443, b=0.000050, MAPE=2342.78%
Log-Linear Supply → a=10.2787, b=-0.000222, MAPE=26.36%


## 🌉 Section 6 — Price Bridging & Lambda Functions

Set `use_bridging = True` in CONFIG to activate.
Then all composed lambdas are assembled for use in EPAM.


In [146]:
# ── Optional Price Bridging ───────────────────────────────────────────
if use_bridging and bridging_master is not None:
    X_bridge = df_fit[[bridging_master]].values
    secondary_price_col = supply_price_col if bridging_master == demand_price_col else demand_price_col
    y_bridge = df_fit[secondary_price_col].values
    bridge_lin,  _, mape_bl = perform_linear_regression(X_bridge, y_bridge)
    bridge_poly, bridge_feat_b, _, _ = perform_polynomial_regression(X_bridge, y_bridge, degree=bridging_degree)
    b_li, b_lc = bridge_lin.intercept_, bridge_lin.coef_[0]
    b_pi, b_pc = bridge_poly.intercept_, bridge_poly.coef_[1:]
    bridge_lin_f  = lambda p: b_li + b_lc * p
    bridge_poly_f = lambda p: b_pi + sum(c*(p**i) for i,c in enumerate(b_pc, 1))
    print(f'Bridging {bridging_master} → {secondary_price_col} | Linear MAPE={mape_bl:.2%}')
else:
    bridge_lin_f = bridge_poly_f = None
    print('Price bridging disabled.')

# ── Raw lambdas ───────────────────────────────────────────────────────
l_d_i, l_d_c = lin_model_d.intercept_, lin_model_d.coef_[0]
l_s_i, l_s_c = lin_model_s.intercept_, lin_model_s.coef_[0]
raw_lin_d = lambda p: l_d_i + l_d_c * p
raw_lin_s = lambda p: l_s_i + l_s_c * p

p_d_i, p_d_c = poly_model_d.intercept_, poly_model_d.coef_[1:]
p_s_i, p_s_c = poly_model_s.intercept_, poly_model_s.coef_[1:]
raw_poly_d = lambda p: p_d_i + sum(c*(p**i) for i,c in enumerate(p_d_c, 1))
raw_poly_s = lambda p: p_s_i + sum(c*(p**i) for i,c in enumerate(p_s_c, 1))

raw_na_d = lambda p, t: na_d_int + na_d_coefs[0]*p + na_d_coefs[1]*t
raw_na_s = lambda p, t: na_s_int + na_s_coefs[0]*p + na_s_coefs[1]*t

ll_d_a, ll_d_b = loglin_model_d.intercept_, loglin_model_d.coef_[0]
ll_s_a, ll_s_b = loglin_model_s.intercept_, loglin_model_s.coef_[0]
raw_loglin_d = lambda p: np.exp(ll_d_a + ll_d_b * p)
raw_loglin_s = lambda p: np.exp(ll_s_a + ll_s_b * p)

# ── Compose with bridging ─────────────────────────────────────────────
if use_bridging and bridging_master is not None:
    if bridging_master == demand_price_col:
        lin_d_func = raw_lin_d; poly_d_func = raw_poly_d; na_d_func = raw_na_d
        loglin_d_func = raw_loglin_d
        lin_s_func = lambda p: raw_lin_s(bridge_lin_f(p))
        poly_s_func = lambda p: raw_poly_s(bridge_poly_f(p))
        na_s_func = lambda p, t: raw_na_s(bridge_lin_f(p), t)
        loglin_s_func = lambda p: raw_loglin_s(bridge_lin_f(p))
    else:
        lin_s_func = raw_lin_s; poly_s_func = raw_poly_s; na_s_func = raw_na_s
        loglin_s_func = raw_loglin_s
        lin_d_func = lambda p: raw_lin_d(bridge_lin_f(p))
        poly_d_func = lambda p: raw_poly_d(bridge_poly_f(p))
        na_d_func = lambda p, t: raw_na_d(bridge_lin_f(p), t)
        loglin_d_func = lambda p: raw_loglin_d(bridge_lin_f(p))
    price_col_main = bridging_master
else:
    lin_d_func = raw_lin_d; lin_s_func = raw_lin_s
    poly_d_func = raw_poly_d; poly_s_func = raw_poly_s
    na_d_func = raw_na_d; na_s_func = raw_na_s
    loglin_d_func = raw_loglin_d; loglin_s_func = raw_loglin_s
    price_col_main = demand_price_col

prices_fit = df_fit[price_col_main].values
prices_fc  = df_forecast_true[price_col_main].values if price_col_main in df_forecast_true.columns else df_forecast_true[demand_price_col].values
print('✅ Lambda functions composed.')


Price bridging disabled.
✅ Lambda functions composed.


## 🔁 Section 7 — EPAM Simulation & Forecasting

For each model, two phases:
- **In-sample**: ODE solved over fit rows → compare vs `prices_fit`
- **Out-of-sample**: ODE continued into held-out rows → compare vs `prices_fc`


## 🔁 Section 7 — EPAM Simulation & Forecasting

Each model is solved under **both** K methods:
- **Constant K**: K fixed at the value derived from t=0
- **Dynamic K**: K(t) computed per eq 4.22 from historical prices, extrapolated linearly for forecast


In [147]:
# ══ LINEAR ══════════════════════════════════════════════════════════════════

# ── Constant K ───────────────────────────────────────────────────────────────
_g0 = lin_d_func(prices_fit[0]) - lin_s_func(prices_fit[0])
lin_k0 = (prices_fit[1]-prices_fit[0])/_g0 if len(prices_fit)>=2 and abs(_g0)>1e-6 else 0.0
lin_k_const_func = lambda t, k=lin_k0: k

lin_P_sim_const = solve_epam(prices_fit[0], lin_k_const_func, lin_d_func, lin_s_func, time_steps_fit)
lin_mape_sim_const = mean_absolute_percentage_error(prices_fit, lin_P_sim_const)
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
lin_P_fc_const = solve_epam(lin_P_sim_const[-1], lin_k_const_func, lin_d_func, lin_s_func, _t_in)[1:]
lin_mape_fc_const = mean_absolute_percentage_error(prices_fc, lin_P_fc_const) if n_forecast>0 and len(lin_P_fc_const)>0 else float('nan')
print(f'Linear Constant K={lin_k0:.6f} | In-sample={lin_mape_sim_const:.2%} | Forecast={lin_mape_fc_const:.2%}')

# ── Dynamic K ─────────────────────────────────────────────────────────────────
lin_k_dyn_vals = []
for i in range(len(prices_fit)-1):
    _pc, _pn = prices_fit[i], prices_fit[i+1]
    _gap = lin_d_func(_pc) - lin_s_func(_pc)
    _kt = (_pn-_pc)/_gap if abs(_gap)>1e-4 else (lin_k_dyn_vals[-1] if lin_k_dyn_vals else 0.0)
    lin_k_dyn_vals.append(_kt)
lin_k_dyn_vals.append(lin_k_dyn_vals[-1] if lin_k_dyn_vals else 0.0)
lin_k_dyn_vals = np.array(lin_k_dyn_vals)
lin_k_dyn_func = interp1d(time_steps_fit, lin_k_dyn_vals, kind='previous', fill_value='extrapolate')

lin_P_sim_dyn = solve_epam(prices_fit[0], lin_k_dyn_func, lin_d_func, lin_s_func, time_steps_fit)
lin_mape_sim_dyn = mean_absolute_percentage_error(prices_fit, lin_P_sim_dyn)
try:    lin_k_slope, lin_k_intercept = np.polyfit(time_steps_fit, np.nan_to_num(lin_k_dyn_vals), 1)
except: lin_k_slope, lin_k_intercept = 0.0, float(lin_k_dyn_vals[-1])
lin_k_dyn_fc_func = lambda t, s=lin_k_slope, ic=lin_k_intercept: s*t + ic
lin_k_dyn_fc_vals = lin_k_slope*time_steps_fc + lin_k_intercept
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
lin_P_fc_dyn = solve_epam(lin_P_sim_dyn[-1], lin_k_dyn_fc_func, lin_d_func, lin_s_func, _t_in)[1:]
lin_mape_fc_dyn = mean_absolute_percentage_error(prices_fc, lin_P_fc_dyn) if n_forecast>0 and len(lin_P_fc_dyn)>0 else float('nan')
print(f'Linear Dynamic K  | In-sample={lin_mape_sim_dyn:.2%} | Forecast={lin_mape_fc_dyn:.2%}')


Linear Constant K=-0.064661 | In-sample=2.46% | Forecast=17.09%
Linear Dynamic K  | In-sample=2.17% | Forecast=5.57%


In [148]:
# ══ POLYNOMIAL ══════════════════════════════════════════════════════════════

# ── Constant K ───────────────────────────────────────────────────────────────
_g0 = poly_d_func(prices_fit[0]) - poly_s_func(prices_fit[0])
poly_k0 = (prices_fit[1]-prices_fit[0])/_g0 if len(prices_fit)>=2 and abs(_g0)>1e-6 else 0.0
poly_k_const_func = lambda t, k=poly_k0: k

poly_P_sim_const = solve_epam(prices_fit[0], poly_k_const_func, poly_d_func, poly_s_func, time_steps_fit)
poly_mape_sim_const = mean_absolute_percentage_error(prices_fit, poly_P_sim_const)
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
poly_P_fc_const = solve_epam(poly_P_sim_const[-1], poly_k_const_func, poly_d_func, poly_s_func, _t_in)[1:]
poly_mape_fc_const = mean_absolute_percentage_error(prices_fc, poly_P_fc_const) if n_forecast>0 and len(poly_P_fc_const)>0 else float('nan')
print(f'Polynomial Constant K={poly_k0:.6f} | In-sample={poly_mape_sim_const:.2%} | Forecast={poly_mape_fc_const:.2%}')

# ── Dynamic K ─────────────────────────────────────────────────────────────────
poly_k_dyn_vals = []
for i in range(len(prices_fit)-1):
    _pc, _pn = prices_fit[i], prices_fit[i+1]
    _gap = poly_d_func(_pc) - poly_s_func(_pc)
    _kt = (_pn-_pc)/_gap if abs(_gap)>1e-4 else (poly_k_dyn_vals[-1] if poly_k_dyn_vals else 0.0)
    poly_k_dyn_vals.append(_kt)
poly_k_dyn_vals.append(poly_k_dyn_vals[-1] if poly_k_dyn_vals else 0.0)
poly_k_dyn_vals = np.array(poly_k_dyn_vals)
poly_k_dyn_func = interp1d(time_steps_fit, poly_k_dyn_vals, kind='previous', fill_value='extrapolate')

poly_P_sim_dyn = solve_epam(prices_fit[0], poly_k_dyn_func, poly_d_func, poly_s_func, time_steps_fit)
poly_mape_sim_dyn = mean_absolute_percentage_error(prices_fit, poly_P_sim_dyn)
try:    poly_k_slope, poly_k_intercept = np.polyfit(time_steps_fit, np.nan_to_num(poly_k_dyn_vals), 1)
except: poly_k_slope, poly_k_intercept = 0.0, float(poly_k_dyn_vals[-1])
poly_k_dyn_fc_func = lambda t, s=poly_k_slope, ic=poly_k_intercept: s*t + ic
poly_k_dyn_fc_vals = poly_k_slope*time_steps_fc + poly_k_intercept
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
poly_P_fc_dyn = solve_epam(poly_P_sim_dyn[-1], poly_k_dyn_fc_func, poly_d_func, poly_s_func, _t_in)[1:]
poly_mape_fc_dyn = mean_absolute_percentage_error(prices_fc, poly_P_fc_dyn) if n_forecast>0 and len(poly_P_fc_dyn)>0 else float('nan')
print(f'Polynomial Dynamic K  | In-sample={poly_mape_sim_dyn:.2%} | Forecast={poly_mape_fc_dyn:.2%}')


Polynomial Constant K=-0.003556 | In-sample=2.35% | Forecast=16.60%
Polynomial Dynamic K  | In-sample=11088.50% | Forecast=33961678.14%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.



In [149]:
# ══ MULTIVARIATE ════════════════════════════════════════════════════════════

# ── Constant K ───────────────────────────────────────────────────────────────
_g0 = na_d_func(prices_fit[0], time_steps_fit[0]) - na_s_func(prices_fit[0], time_steps_fit[0])
na_k0 = (prices_fit[1]-prices_fit[0])/_g0 if len(prices_fit)>=2 and abs(_g0)>1e-6 else 0.0
na_k_const_func = lambda t, k=na_k0: k

na_P_sim_const = solve_non_autonomous_epam(prices_fit[0], na_k_const_func, na_d_func, na_s_func, time_steps_fit)
na_mape_sim_const = mean_absolute_percentage_error(prices_fit, na_P_sim_const)
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
na_P_fc_const = solve_non_autonomous_epam(na_P_sim_const[-1], na_k_const_func, na_d_func, na_s_func, _t_in)[1:]
na_mape_fc_const = mean_absolute_percentage_error(prices_fc, na_P_fc_const) if n_forecast>0 and len(na_P_fc_const)>0 else float('nan')
print(f'Multivariate Constant K={na_k0:.6f} | In-sample={na_mape_sim_const:.2%} | Forecast={na_mape_fc_const:.2%}')

# ── Dynamic K ─────────────────────────────────────────────────────────────────
na_k_dyn_vals = []
for i in range(len(prices_fit)-1):
    _pc, _pn, _tc = prices_fit[i], prices_fit[i+1], time_steps_fit[i]
    _gap = na_d_func(_pc, _tc) - na_s_func(_pc, _tc)
    _kt = (_pn-_pc)/_gap if abs(_gap)>1e-4 else (na_k_dyn_vals[-1] if na_k_dyn_vals else 0.0)
    na_k_dyn_vals.append(_kt)
na_k_dyn_vals.append(na_k_dyn_vals[-1] if na_k_dyn_vals else 0.0)
na_k_dyn_vals = np.array(na_k_dyn_vals)
na_k_dyn_func = interp1d(time_steps_fit, na_k_dyn_vals, kind='previous', fill_value='extrapolate')

na_P_sim_dyn = solve_non_autonomous_epam(prices_fit[0], na_k_dyn_func, na_d_func, na_s_func, time_steps_fit)
na_mape_sim_dyn = mean_absolute_percentage_error(prices_fit, na_P_sim_dyn)
try:    na_k_slope, na_k_intercept = np.polyfit(time_steps_fit, np.nan_to_num(na_k_dyn_vals), 1)
except: na_k_slope, na_k_intercept = 0.0, float(na_k_dyn_vals[-1])
na_k_dyn_fc_func = lambda t, s=na_k_slope, ic=na_k_intercept: s*t + ic
na_k_dyn_fc_vals = na_k_slope*time_steps_fc + na_k_intercept
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
na_P_fc_dyn = solve_non_autonomous_epam(na_P_sim_dyn[-1], na_k_dyn_fc_func, na_d_func, na_s_func, _t_in)[1:]
na_mape_fc_dyn = mean_absolute_percentage_error(prices_fc, na_P_fc_dyn) if n_forecast>0 and len(na_P_fc_dyn)>0 else float('nan')
print(f'Multivariate Dynamic K  | In-sample={na_mape_sim_dyn:.2%} | Forecast={na_mape_fc_dyn:.2%}')


Multivariate Constant K=-0.030677 | In-sample=2.88% | Forecast=19.02%
Multivariate Dynamic K  | In-sample=1.25% | Forecast=3.30%


In [150]:
# ══ LOG-LINEAR ══════════════════════════════════════════════════════════════

# ── Constant K ───────────────────────────────────────────────────────────────
_g0 = loglin_d_func(prices_fit[0]) - loglin_s_func(prices_fit[0])
ll_k0 = (prices_fit[1]-prices_fit[0])/_g0 if len(prices_fit)>=2 and abs(_g0)>1e-6 else 0.0
ll_k_const_func = lambda t, k=ll_k0: k

ll_P_sim_const = solve_epam(prices_fit[0], ll_k_const_func, loglin_d_func, loglin_s_func, time_steps_fit)
ll_mape_sim_const = mean_absolute_percentage_error(prices_fit, ll_P_sim_const)
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
ll_P_fc_const = solve_epam(ll_P_sim_const[-1], ll_k_const_func, loglin_d_func, loglin_s_func, _t_in)[1:]
ll_mape_fc_const = mean_absolute_percentage_error(prices_fc, ll_P_fc_const) if n_forecast>0 and len(ll_P_fc_const)>0 else float('nan')
print(f'Log-Linear Constant K={ll_k0:.6f} | In-sample={ll_mape_sim_const:.2%} | Forecast={ll_mape_fc_const:.2%}')

# ── Dynamic K ─────────────────────────────────────────────────────────────────
ll_k_dyn_vals = []
for i in range(len(prices_fit)-1):
    _pc, _pn = prices_fit[i], prices_fit[i+1]
    _gap = loglin_d_func(_pc) - loglin_s_func(_pc)
    _kt = (_pn-_pc)/_gap if abs(_gap)>1e-4 else (ll_k_dyn_vals[-1] if ll_k_dyn_vals else 0.0)
    ll_k_dyn_vals.append(_kt)
ll_k_dyn_vals.append(ll_k_dyn_vals[-1] if ll_k_dyn_vals else 0.0)
ll_k_dyn_vals = np.array(ll_k_dyn_vals)
ll_k_dyn_func = interp1d(time_steps_fit, ll_k_dyn_vals, kind='previous', fill_value='extrapolate')

ll_P_sim_dyn = solve_epam(prices_fit[0], ll_k_dyn_func, loglin_d_func, loglin_s_func, time_steps_fit)
ll_mape_sim_dyn = mean_absolute_percentage_error(prices_fit, ll_P_sim_dyn)
try:    ll_k_slope, ll_k_intercept = np.polyfit(time_steps_fit, np.nan_to_num(ll_k_dyn_vals), 1)
except: ll_k_slope, ll_k_intercept = 0.0, float(ll_k_dyn_vals[-1])
ll_k_dyn_fc_func = lambda t, s=ll_k_slope, ic=ll_k_intercept: s*t + ic
ll_k_dyn_fc_vals = ll_k_slope*time_steps_fc + ll_k_intercept
_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
ll_P_fc_dyn = solve_epam(ll_P_sim_dyn[-1], ll_k_dyn_fc_func, loglin_d_func, loglin_s_func, _t_in)[1:]
ll_mape_fc_dyn = mean_absolute_percentage_error(prices_fc, ll_P_fc_dyn) if n_forecast>0 and len(ll_P_fc_dyn)>0 else float('nan')
print(f'Log-Linear Dynamic K  | In-sample={ll_mape_sim_dyn:.2%} | Forecast={ll_mape_fc_dyn:.2%}')


Log-Linear Constant K=-0.220039 | In-sample=2.18% | Forecast=15.82%
Log-Linear Dynamic K  | In-sample=296.33% | Forecast=20.43%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.



## 🔄 Section 7b — Rolling Window K EPAM

K(t) estimated via expanding→rolling OLS **without intercept** (`K = β·t`).
At each time step the regression uses the last `rolling_k_window` raw K values.
Forecast K values are produced iteratively: predict next K, append, slide window.


In [151]:
# ══ LINEAR — Rolling Window K ═══════════════════════════════════════════════

lin_k_roll_vals, lin_k_roll_last_beta = compute_rolling_k(
    lin_k_dyn_vals, time_steps_fit, rolling_k_window)
lin_k_roll_func = interp1d(time_steps_fit, lin_k_roll_vals,
                           kind='previous', fill_value='extrapolate')

lin_P_sim_roll = solve_epam(prices_fit[0], lin_k_roll_func,
                           lin_d_func, lin_s_func, time_steps_fit)
lin_mape_sim_roll = mean_absolute_percentage_error(prices_fit, lin_P_sim_roll)

# Forecast K iteratively
lin_k_roll_fc_vals = forecast_rolling_k(
    lin_k_dyn_vals, time_steps_fit, time_steps_fc, rolling_k_window)
lin_k_roll_fc_func = interp1d(
    np.concatenate([time_steps_fit, time_steps_fc]),
    np.concatenate([lin_k_roll_vals, lin_k_roll_fc_vals]),
    kind='previous', fill_value='extrapolate')

_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
lin_P_fc_roll = solve_epam(lin_P_sim_roll[-1], lin_k_roll_fc_func,
                          lin_d_func, lin_s_func, _t_in)[1:]
lin_mape_fc_roll = mean_absolute_percentage_error(
    prices_fc, lin_P_fc_roll) if n_forecast > 0 and len(lin_P_fc_roll) > 0 else float('nan')
print(f'Linear Rolling K (w={rolling_k_window}) '
      f'| In-sample={lin_mape_sim_roll:.2%} | Forecast={lin_mape_fc_roll:.2%}')


Linear Rolling K (w=12) | In-sample=7.96% | Forecast=30.96%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29:

In [152]:
# ══ POLYNOMIAL — Rolling Window K ═══════════════════════════════════════════

poly_k_roll_vals, poly_k_roll_last_beta = compute_rolling_k(
    poly_k_dyn_vals, time_steps_fit, rolling_k_window)
poly_k_roll_func = interp1d(time_steps_fit, poly_k_roll_vals,
                            kind='previous', fill_value='extrapolate')

poly_P_sim_roll = solve_epam(prices_fit[0], poly_k_roll_func,
                            poly_d_func, poly_s_func, time_steps_fit)
poly_mape_sim_roll = mean_absolute_percentage_error(prices_fit, poly_P_sim_roll)

# Forecast K iteratively
poly_k_roll_fc_vals = forecast_rolling_k(
    poly_k_dyn_vals, time_steps_fit, time_steps_fc, rolling_k_window)
poly_k_roll_fc_func = interp1d(
    np.concatenate([time_steps_fit, time_steps_fc]),
    np.concatenate([poly_k_roll_vals, poly_k_roll_fc_vals]),
    kind='previous', fill_value='extrapolate')

_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
poly_P_fc_roll = solve_epam(poly_P_sim_roll[-1], poly_k_roll_fc_func,
                           poly_d_func, poly_s_func, _t_in)[1:]
poly_mape_fc_roll = mean_absolute_percentage_error(
    prices_fc, poly_P_fc_roll) if n_forecast > 0 and len(poly_P_fc_roll) > 0 else float('nan')
print(f'Polynomial Rolling K (w={rolling_k_window}) '
      f'| In-sample={poly_mape_sim_roll:.2%} | Forecast={poly_mape_fc_roll:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and w

Polynomial Rolling K (w=12) | In-sample=17445.07% | Forecast=36791921.21%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.



In [153]:
# ══ MULTIVARIATE — Rolling Window K ═════════════════════════════════════════

na_k_roll_vals, na_k_roll_last_beta = compute_rolling_k(
    na_k_dyn_vals, time_steps_fit, rolling_k_window)
na_k_roll_func = interp1d(time_steps_fit, na_k_roll_vals,
                          kind='previous', fill_value='extrapolate')

na_P_sim_roll = solve_non_autonomous_epam(prices_fit[0], na_k_roll_func,
                                         na_d_func, na_s_func, time_steps_fit)
na_mape_sim_roll = mean_absolute_percentage_error(prices_fit, na_P_sim_roll)

# Forecast K iteratively
na_k_roll_fc_vals = forecast_rolling_k(
    na_k_dyn_vals, time_steps_fit, time_steps_fc, rolling_k_window)
na_k_roll_fc_func = interp1d(
    np.concatenate([time_steps_fit, time_steps_fc]),
    np.concatenate([na_k_roll_vals, na_k_roll_fc_vals]),
    kind='previous', fill_value='extrapolate')

_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
na_P_fc_roll = solve_non_autonomous_epam(na_P_sim_roll[-1], na_k_roll_fc_func,
                                        na_d_func, na_s_func, _t_in)[1:]
na_mape_fc_roll = mean_absolute_percentage_error(
    prices_fc, na_P_fc_roll) if n_forecast > 0 and len(na_P_fc_roll) > 0 else float('nan')
print(f'Multivariate Rolling K (w={rolling_k_window}) '
      f'| In-sample={na_mape_sim_roll:.2%} | Forecast={na_mape_fc_roll:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Multivariate Rolling K (w=12) | In-sample=1.17% | Forecast=5.34%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



In [154]:
# ══ LOG-LINEAR — Rolling Window K ════════════════════════════════════════════

ll_k_roll_vals, ll_k_roll_last_beta = compute_rolling_k(
    ll_k_dyn_vals, time_steps_fit, rolling_k_window)
ll_k_roll_func = interp1d(time_steps_fit, ll_k_roll_vals,
                          kind='previous', fill_value='extrapolate')

ll_P_sim_roll = solve_epam(prices_fit[0], ll_k_roll_func,
                          loglin_d_func, loglin_s_func, time_steps_fit)
ll_mape_sim_roll = mean_absolute_percentage_error(prices_fit, ll_P_sim_roll)

ll_k_roll_fc_vals = forecast_rolling_k(
    ll_k_dyn_vals, time_steps_fit, time_steps_fc, rolling_k_window)
ll_k_roll_fc_func = interp1d(
    np.concatenate([time_steps_fit, time_steps_fc]),
    np.concatenate([ll_k_roll_vals, ll_k_roll_fc_vals]),
    kind='previous', fill_value='extrapolate')

_t_in = np.concatenate(([time_steps_fit[-1]], time_steps_fc))
ll_P_fc_roll = solve_epam(ll_P_sim_roll[-1], ll_k_roll_fc_func,
                         loglin_d_func, loglin_s_func, _t_in)[1:]
ll_mape_fc_roll = mean_absolute_percentage_error(
    prices_fc, ll_P_fc_roll) if n_forecast > 0 and len(ll_P_fc_roll) > 0 else float('nan')
print(f'Log-Linear Rolling K (w={rolling_k_window}) '
      f'| In-sample={ll_mape_sim_roll:.2%} | Forecast={ll_mape_fc_roll:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Log-Linear Rolling K (w=12) | In-sample=338.53% | Forecast=215.38%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:79: ODEintWarning:

Excess work done on this call (perhaps wrong Dfun type). Run with full_out

## 🔄 Section 7c — Adjusted Gap EPAM

Gap dihitung dari prediksi regresi pada harga aktual, kemudian bias struktural
(Mean Gap / buffer stock) dikurangi. Setiap kombinasi K dijalankan menggunakan solusi kontinu (ODE).

$$Gap_{adj,t} = (Q_{d,t} - Q_{s,t}) - \overline{Gap}$$
$$P_{t+1} = P_t + k_t \cdot Gap_{adj,t} \quad \text{(discrete)}$$
$$\frac{dP}{dt} = k(t) \cdot Gap_{adj}(t) \quad \text{(continuous)}$$


In [155]:
# ══ LINEAR — Adjusted Gap ═══════════════════════════════════════════

lin_ag_Qd = np.array([lin_d_func(p) for p in prices_fit])
lin_ag_Qs = np.array([lin_s_func(p) for p in prices_fit])
lin_ag_gap_raw, lin_ag_gap_mean, lin_ag_gap_adj = compute_adjusted_gap(lin_ag_Qd, lin_ag_Qs)
delta_P = np.diff(prices_fit)

# ── Constant K ───────────────────────────────────────────────────────
lin_ag_k0 = delta_P[0] / lin_ag_gap_adj[0] if abs(lin_ag_gap_adj[0]) > 1e-6 else 0.0

# Continuous
lin_ag_k0_func = lambda t, k=lin_ag_k0: k
lin_ag_P_const_cont = solve_epam_adjusted_gap(prices_fit[0], lin_ag_k0_func, lin_ag_gap_adj, time_steps_fit)
lin_ag_mape_const_cont = mean_absolute_percentage_error(prices_fit, lin_ag_P_const_cont)

# ── Dynamic K ────────────────────────────────────────────────────────
lin_ag_k_dyn = np.array([
    dp / g if abs(g) > 1e-4 else (lin_ag_k0)
    for dp, g in zip(delta_P, lin_ag_gap_adj[:-1])
])
lin_ag_k_dyn = np.append(lin_ag_k_dyn, lin_ag_k_dyn[-1])

# Continuous
lin_ag_k_dyn_func = interp1d(time_steps_fit, lin_ag_k_dyn, kind='previous', fill_value='extrapolate')
lin_ag_P_dyn_cont = solve_epam_adjusted_gap(prices_fit[0], lin_ag_k_dyn_func, lin_ag_gap_adj, time_steps_fit)
lin_ag_mape_dyn_cont = mean_absolute_percentage_error(prices_fit, lin_ag_P_dyn_cont)

# ── Rolling K ────────────────────────────────────────────────────────
lin_ag_k_roll, lin_ag_k_roll_beta = compute_rolling_k(
    lin_ag_k_dyn, time_steps_fit, rolling_k_window)

# Continuous
lin_ag_k_roll_func = interp1d(time_steps_fit, lin_ag_k_roll, kind='previous', fill_value='extrapolate')
lin_ag_P_roll_cont = solve_epam_adjusted_gap(prices_fit[0], lin_ag_k_roll_func, lin_ag_gap_adj, time_steps_fit)
lin_ag_mape_roll_cont = mean_absolute_percentage_error(prices_fit, lin_ag_P_roll_cont)

# ── Forecast ─────────────────────────────────────────────────────────
if n_forecast > 0:
    _Qd_fc = np.array([lin_d_func(p) for p in prices_fc])
    _Qs_fc = np.array([lin_s_func(p) for p in prices_fc])
    _gap_fc = (_Qd_fc - _Qs_fc) - lin_ag_gap_mean

    # Constant K forecast
    lin_ag_P_fc_const_cont = solve_epam_adjusted_gap(lin_ag_P_const_cont[-1], lin_ag_k0_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Dynamic K forecast — extrapolate linearly
    try:    lin_ag_ks, lin_ag_ki = np.polyfit(time_steps_fit, np.nan_to_num(lin_ag_k_dyn), 1)
    except: lin_ag_ks, lin_ag_ki = 0.0, float(lin_ag_k_dyn[-1])
    lin_ag_k_dyn_fc_func = lambda t, s=lin_ag_ks, ic=lin_ag_ki: s*t + ic
    lin_ag_k_dyn_fc_vals = lin_ag_ks*time_steps_fc + lin_ag_ki
    lin_ag_P_fc_dyn_cont = solve_epam_adjusted_gap(lin_ag_P_dyn_cont[-1], lin_ag_k_dyn_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Rolling K forecast — iterative
    lin_ag_k_roll_fc = forecast_rolling_k(lin_ag_k_dyn, time_steps_fit, time_steps_fc, rolling_k_window)
    _all_k = np.concatenate([lin_ag_k_roll, lin_ag_k_roll_fc])
    _all_t = np.concatenate([time_steps_fit, time_steps_fc])
    lin_ag_k_roll_fc_func = interp1d(_all_t, _all_k, kind='previous', fill_value='extrapolate')
    lin_ag_P_fc_roll_cont = solve_epam_adjusted_gap(lin_ag_P_roll_cont[-1], lin_ag_k_roll_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])
else:
    lin_ag_P_fc_const_cont = np.array([])
    lin_ag_P_fc_dyn_cont   = np.array([])
    lin_ag_P_fc_roll_cont  = np.array([])

lin_ag_mape_fc_const_cont = mean_absolute_percentage_error(prices_fc, lin_ag_P_fc_const_cont) if n_forecast > 0 and len(lin_ag_P_fc_const_cont) > 0 else float('nan')
lin_ag_mape_fc_dyn_cont = mean_absolute_percentage_error(prices_fc, lin_ag_P_fc_dyn_cont) if n_forecast > 0 and len(lin_ag_P_fc_dyn_cont) > 0 else float('nan')
lin_ag_mape_fc_roll_cont = mean_absolute_percentage_error(prices_fc, lin_ag_P_fc_roll_cont) if n_forecast > 0 and len(lin_ag_P_fc_roll_cont) > 0 else float('nan')

print(f'Linear Adjusted Gap (Mean Gap={lin_ag_gap_mean:.4f})')
print(f'  Constant K={lin_ag_k0:.6f}')
print(f'    Continuous: In-sample={lin_ag_mape_const_cont:.2%}   Forecast={lin_ag_mape_fc_const_cont:.2%}')
print(f'  Dynamic K')
print(f'    Continuous: In-sample={lin_ag_mape_dyn_cont:.2%}   Forecast={lin_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={lin_ag_k_roll_beta:.6f}')
print(f'    Continuous: In-sample={lin_ag_mape_roll_cont:.2%}   Forecast={lin_ag_mape_fc_roll_cont:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29:

Linear Adjusted Gap (Mean Gap=487.3964)
  Constant K=0.000000
    Continuous: In-sample=1.50%   Forecast=13.41%
  Dynamic K
    Continuous: In-sample=1.44%   Forecast=87.36%
  Rolling K (w=12), last β=0.014863
    Continuous: In-sample=15.21%   Forecast=18.05%


In [156]:
# ══ POLYNOMIAL — Adjusted Gap ═══════════════════════════════════════════

poly_ag_Qd = np.array([poly_d_func(p) for p in prices_fit])
poly_ag_Qs = np.array([poly_s_func(p) for p in prices_fit])
poly_ag_gap_raw, poly_ag_gap_mean, poly_ag_gap_adj = compute_adjusted_gap(poly_ag_Qd, poly_ag_Qs)
delta_P = np.diff(prices_fit)

# ── Constant K ───────────────────────────────────────────────────────
poly_ag_k0 = delta_P[0] / poly_ag_gap_adj[0] if abs(poly_ag_gap_adj[0]) > 1e-6 else 0.0

# Continuous
poly_ag_k0_func = lambda t, k=poly_ag_k0: k
poly_ag_P_const_cont = solve_epam_adjusted_gap(prices_fit[0], poly_ag_k0_func, poly_ag_gap_adj, time_steps_fit)
poly_ag_mape_const_cont = mean_absolute_percentage_error(prices_fit, poly_ag_P_const_cont)

# ── Dynamic K ────────────────────────────────────────────────────────
poly_ag_k_dyn = np.array([
    dp / g if abs(g) > 1e-4 else (poly_ag_k0)
    for dp, g in zip(delta_P, poly_ag_gap_adj[:-1])
])
poly_ag_k_dyn = np.append(poly_ag_k_dyn, poly_ag_k_dyn[-1])

# Continuous
poly_ag_k_dyn_func = interp1d(time_steps_fit, poly_ag_k_dyn, kind='previous', fill_value='extrapolate')
poly_ag_P_dyn_cont = solve_epam_adjusted_gap(prices_fit[0], poly_ag_k_dyn_func, poly_ag_gap_adj, time_steps_fit)
poly_ag_mape_dyn_cont = mean_absolute_percentage_error(prices_fit, poly_ag_P_dyn_cont)

# ── Rolling K ────────────────────────────────────────────────────────
poly_ag_k_roll, poly_ag_k_roll_beta = compute_rolling_k(
    poly_ag_k_dyn, time_steps_fit, rolling_k_window)

# Continuous
poly_ag_k_roll_func = interp1d(time_steps_fit, poly_ag_k_roll, kind='previous', fill_value='extrapolate')
poly_ag_P_roll_cont = solve_epam_adjusted_gap(prices_fit[0], poly_ag_k_roll_func, poly_ag_gap_adj, time_steps_fit)
poly_ag_mape_roll_cont = mean_absolute_percentage_error(prices_fit, poly_ag_P_roll_cont)

# ── Forecast ─────────────────────────────────────────────────────────
if n_forecast > 0:
    _Qd_fc = np.array([poly_d_func(p) for p in prices_fc])
    _Qs_fc = np.array([poly_s_func(p) for p in prices_fc])
    _gap_fc = (_Qd_fc - _Qs_fc) - poly_ag_gap_mean

    # Constant K forecast
    poly_ag_P_fc_const_cont = solve_epam_adjusted_gap(poly_ag_P_const_cont[-1], poly_ag_k0_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Dynamic K forecast — extrapolate linearly
    try:    poly_ag_ks, poly_ag_ki = np.polyfit(time_steps_fit, np.nan_to_num(poly_ag_k_dyn), 1)
    except: poly_ag_ks, poly_ag_ki = 0.0, float(poly_ag_k_dyn[-1])
    poly_ag_k_dyn_fc_func = lambda t, s=poly_ag_ks, ic=poly_ag_ki: s*t + ic
    poly_ag_k_dyn_fc_vals = poly_ag_ks*time_steps_fc + poly_ag_ki
    poly_ag_P_fc_dyn_cont = solve_epam_adjusted_gap(poly_ag_P_dyn_cont[-1], poly_ag_k_dyn_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Rolling K forecast — iterative
    poly_ag_k_roll_fc = forecast_rolling_k(poly_ag_k_dyn, time_steps_fit, time_steps_fc, rolling_k_window)
    _all_k = np.concatenate([poly_ag_k_roll, poly_ag_k_roll_fc])
    _all_t = np.concatenate([time_steps_fit, time_steps_fc])
    poly_ag_k_roll_fc_func = interp1d(_all_t, _all_k, kind='previous', fill_value='extrapolate')
    poly_ag_P_fc_roll_cont = solve_epam_adjusted_gap(poly_ag_P_roll_cont[-1], poly_ag_k_roll_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])
else:
    poly_ag_P_fc_const_cont = np.array([])
    poly_ag_P_fc_dyn_cont   = np.array([])
    poly_ag_P_fc_roll_cont  = np.array([])

poly_ag_mape_fc_const_cont = mean_absolute_percentage_error(prices_fc, poly_ag_P_fc_const_cont) if n_forecast > 0 and len(poly_ag_P_fc_const_cont) > 0 else float('nan')
poly_ag_mape_fc_dyn_cont = mean_absolute_percentage_error(prices_fc, poly_ag_P_fc_dyn_cont) if n_forecast > 0 and len(poly_ag_P_fc_dyn_cont) > 0 else float('nan')
poly_ag_mape_fc_roll_cont = mean_absolute_percentage_error(prices_fc, poly_ag_P_fc_roll_cont) if n_forecast > 0 and len(poly_ag_P_fc_roll_cont) > 0 else float('nan')

print(f'Polynomial Adjusted Gap (Mean Gap={poly_ag_gap_mean:.4f})')
print(f'  Constant K={poly_ag_k0:.6f}')
print(f'    Continuous: In-sample={poly_ag_mape_const_cont:.2%}   Forecast={poly_ag_mape_fc_const_cont:.2%}')
print(f'  Dynamic K')
print(f'    Continuous: In-sample={poly_ag_mape_dyn_cont:.2%}   Forecast={poly_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={poly_ag_k_roll_beta:.6f}')
print(f'    Continuous: In-sample={poly_ag_mape_roll_cont:.2%}   Forecast={poly_ag_mape_fc_roll_cont:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Polynomial Adjusted Gap (Mean Gap=9294.6800)
  Constant K=0.072984
    Continuous: In-sample=8.25%   Forecast=213.98%
  Dynamic K
    Continuous: In-sample=1.54%   Forecast=1348.22%
  Rolling K (w=12), last β=-0.006391
    Continuous: In-sample=36.88%   Forecast=833.07%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



In [157]:
# ══ MULTIVARIATE — Adjusted Gap ═══════════════════════════════════════════

na_ag_Qd = np.array([na_d_func(p, t) for p, t in zip(prices_fit, time_steps_fit)])
na_ag_Qs = np.array([na_s_func(p, t) for p, t in zip(prices_fit, time_steps_fit)])
na_ag_gap_raw, na_ag_gap_mean, na_ag_gap_adj = compute_adjusted_gap(na_ag_Qd, na_ag_Qs)
delta_P = np.diff(prices_fit)

# ── Constant K ───────────────────────────────────────────────────────
na_ag_k0 = delta_P[0] / na_ag_gap_adj[0] if abs(na_ag_gap_adj[0]) > 1e-6 else 0.0

# Continuous
na_ag_k0_func = lambda t, k=na_ag_k0: k
na_ag_P_const_cont = solve_epam_adjusted_gap(prices_fit[0], na_ag_k0_func, na_ag_gap_adj, time_steps_fit)
na_ag_mape_const_cont = mean_absolute_percentage_error(prices_fit, na_ag_P_const_cont)

# ── Dynamic K ────────────────────────────────────────────────────────
na_ag_k_dyn = np.array([
    dp / g if abs(g) > 1e-4 else (na_ag_k0)
    for dp, g in zip(delta_P, na_ag_gap_adj[:-1])
])
na_ag_k_dyn = np.append(na_ag_k_dyn, na_ag_k_dyn[-1])

# Continuous
na_ag_k_dyn_func = interp1d(time_steps_fit, na_ag_k_dyn, kind='previous', fill_value='extrapolate')
na_ag_P_dyn_cont = solve_epam_adjusted_gap(prices_fit[0], na_ag_k_dyn_func, na_ag_gap_adj, time_steps_fit)
na_ag_mape_dyn_cont = mean_absolute_percentage_error(prices_fit, na_ag_P_dyn_cont)

# ── Rolling K ────────────────────────────────────────────────────────
na_ag_k_roll, na_ag_k_roll_beta = compute_rolling_k(
    na_ag_k_dyn, time_steps_fit, rolling_k_window)

# Continuous
na_ag_k_roll_func = interp1d(time_steps_fit, na_ag_k_roll, kind='previous', fill_value='extrapolate')
na_ag_P_roll_cont = solve_epam_adjusted_gap(prices_fit[0], na_ag_k_roll_func, na_ag_gap_adj, time_steps_fit)
na_ag_mape_roll_cont = mean_absolute_percentage_error(prices_fit, na_ag_P_roll_cont)

# ── Forecast ─────────────────────────────────────────────────────────
if n_forecast > 0:
    _Qd_fc = np.array([na_d_func(p, t) for p, t in zip(prices_fc, time_steps_fc)])
    _Qs_fc = np.array([na_s_func(p, t) for p, t in zip(prices_fc, time_steps_fc)])
    _gap_fc = (_Qd_fc - _Qs_fc) - na_ag_gap_mean

    # Constant K forecast
    na_ag_P_fc_const_cont = solve_epam_adjusted_gap(na_ag_P_const_cont[-1], na_ag_k0_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Dynamic K forecast — extrapolate linearly
    try:    na_ag_ks, na_ag_ki = np.polyfit(time_steps_fit, np.nan_to_num(na_ag_k_dyn), 1)
    except: na_ag_ks, na_ag_ki = 0.0, float(na_ag_k_dyn[-1])
    na_ag_k_dyn_fc_func = lambda t, s=na_ag_ks, ic=na_ag_ki: s*t + ic
    na_ag_k_dyn_fc_vals = na_ag_ks*time_steps_fc + na_ag_ki
    na_ag_P_fc_dyn_cont = solve_epam_adjusted_gap(na_ag_P_dyn_cont[-1], na_ag_k_dyn_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Rolling K forecast — iterative
    na_ag_k_roll_fc = forecast_rolling_k(na_ag_k_dyn, time_steps_fit, time_steps_fc, rolling_k_window)
    _all_k = np.concatenate([na_ag_k_roll, na_ag_k_roll_fc])
    _all_t = np.concatenate([time_steps_fit, time_steps_fc])
    na_ag_k_roll_fc_func = interp1d(_all_t, _all_k, kind='previous', fill_value='extrapolate')
    na_ag_P_fc_roll_cont = solve_epam_adjusted_gap(na_ag_P_roll_cont[-1], na_ag_k_roll_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])
else:
    na_ag_P_fc_const_cont = np.array([])
    na_ag_P_fc_dyn_cont   = np.array([])
    na_ag_P_fc_roll_cont  = np.array([])

na_ag_mape_fc_const_cont = mean_absolute_percentage_error(prices_fc, na_ag_P_fc_const_cont) if n_forecast > 0 and len(na_ag_P_fc_const_cont) > 0 else float('nan')
na_ag_mape_fc_dyn_cont = mean_absolute_percentage_error(prices_fc, na_ag_P_fc_dyn_cont) if n_forecast > 0 and len(na_ag_P_fc_dyn_cont) > 0 else float('nan')
na_ag_mape_fc_roll_cont = mean_absolute_percentage_error(prices_fc, na_ag_P_fc_roll_cont) if n_forecast > 0 and len(na_ag_P_fc_roll_cont) > 0 else float('nan')

print(f'Multivariate Adjusted Gap (Mean Gap={na_ag_gap_mean:.4f})')
print(f'  Constant K={na_ag_k0:.6f}')
print(f'    Continuous: In-sample={na_ag_mape_const_cont:.2%}   Forecast={na_ag_mape_fc_const_cont:.2%}')
print(f'  Dynamic K')
print(f'    Continuous: In-sample={na_ag_mape_dyn_cont:.2%}   Forecast={na_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={na_ag_k_roll_beta:.6f}')
print(f'    Continuous: In-sample={na_ag_mape_roll_cont:.2%}   Forecast={na_ag_mape_fc_roll_cont:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Multivariate Adjusted Gap (Mean Gap=1176.8400)
  Constant K=0.210804
    Continuous: In-sample=1.78%   Forecast=9.41%
  Dynamic K
    Continuous: In-sample=1.32%   Forecast=5.83%
  Rolling K (w=12), last β=0.018791
    Continuous: In-sample=6.55%   Forecast=11.87%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



In [158]:
# ══ LOG-LINEAR — Adjusted Gap ═══════════════════════════════════════════

ll_ag_Qd = np.array([loglin_d_func(p) for p in prices_fit])
ll_ag_Qs = np.array([loglin_s_func(p) for p in prices_fit])
ll_ag_gap_raw, ll_ag_gap_mean, ll_ag_gap_adj = compute_adjusted_gap(ll_ag_Qd, ll_ag_Qs)
delta_P = np.diff(prices_fit)

# ── Constant K ───────────────────────────────────────────────────────
ll_ag_k0 = delta_P[0] / ll_ag_gap_adj[0] if abs(ll_ag_gap_adj[0]) > 1e-6 else 0.0

# Continuous
ll_ag_k0_func = lambda t, k=ll_ag_k0: k
ll_ag_P_const_cont = solve_epam_adjusted_gap(prices_fit[0], ll_ag_k0_func, ll_ag_gap_adj, time_steps_fit)
ll_ag_mape_const_cont = mean_absolute_percentage_error(prices_fit, ll_ag_P_const_cont)

# ── Dynamic K ────────────────────────────────────────────────────────
ll_ag_k_dyn = np.array([
    dp / g if abs(g) > 1e-4 else (ll_ag_k0)
    for dp, g in zip(delta_P, ll_ag_gap_adj[:-1])
])
ll_ag_k_dyn = np.append(ll_ag_k_dyn, ll_ag_k_dyn[-1])

# Continuous
ll_ag_k_dyn_func = interp1d(time_steps_fit, ll_ag_k_dyn, kind='previous', fill_value='extrapolate')
ll_ag_P_dyn_cont = solve_epam_adjusted_gap(prices_fit[0], ll_ag_k_dyn_func, ll_ag_gap_adj, time_steps_fit)
ll_ag_mape_dyn_cont = mean_absolute_percentage_error(prices_fit, ll_ag_P_dyn_cont)

# ── Rolling K ────────────────────────────────────────────────────────
ll_ag_k_roll, ll_ag_k_roll_beta = compute_rolling_k(
    ll_ag_k_dyn, time_steps_fit, rolling_k_window)

# Continuous
ll_ag_k_roll_func = interp1d(time_steps_fit, ll_ag_k_roll, kind='previous', fill_value='extrapolate')
ll_ag_P_roll_cont = solve_epam_adjusted_gap(prices_fit[0], ll_ag_k_roll_func, ll_ag_gap_adj, time_steps_fit)
ll_ag_mape_roll_cont = mean_absolute_percentage_error(prices_fit, ll_ag_P_roll_cont)

# ── Forecast ─────────────────────────────────────────────────────────
if n_forecast > 0:
    _Qd_fc = np.array([loglin_d_func(p) for p in prices_fc])
    _Qs_fc = np.array([loglin_s_func(p) for p in prices_fc])
    _gap_fc = (_Qd_fc - _Qs_fc) - ll_ag_gap_mean

    # Constant K forecast
    ll_ag_P_fc_const_cont = solve_epam_adjusted_gap(ll_ag_P_const_cont[-1], ll_ag_k0_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Dynamic K forecast — extrapolate linearly
    try:    ll_ag_ks, ll_ag_ki = np.polyfit(time_steps_fit, np.nan_to_num(ll_ag_k_dyn), 1)
    except: ll_ag_ks, ll_ag_ki = 0.0, float(ll_ag_k_dyn[-1])
    ll_ag_k_dyn_fc_func = lambda t, s=ll_ag_ks, ic=ll_ag_ki: s*t + ic
    ll_ag_k_dyn_fc_vals = ll_ag_ks*time_steps_fc + ll_ag_ki
    ll_ag_P_fc_dyn_cont = solve_epam_adjusted_gap(ll_ag_P_dyn_cont[-1], ll_ag_k_dyn_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])

    # Rolling K forecast — iterative
    ll_ag_k_roll_fc = forecast_rolling_k(ll_ag_k_dyn, time_steps_fit, time_steps_fc, rolling_k_window)
    _all_k = np.concatenate([ll_ag_k_roll, ll_ag_k_roll_fc])
    _all_t = np.concatenate([time_steps_fit, time_steps_fc])
    ll_ag_k_roll_fc_func = interp1d(_all_t, _all_k, kind='previous', fill_value='extrapolate')
    ll_ag_P_fc_roll_cont = solve_epam_adjusted_gap(ll_ag_P_roll_cont[-1], ll_ag_k_roll_fc_func, _gap_fc, time_steps_fc) if len(time_steps_fc) > 0 else np.array([])
else:
    ll_ag_P_fc_const_cont = np.array([])
    ll_ag_P_fc_dyn_cont   = np.array([])
    ll_ag_P_fc_roll_cont  = np.array([])

ll_ag_mape_fc_const_cont = mean_absolute_percentage_error(prices_fc, ll_ag_P_fc_const_cont) if n_forecast > 0 and len(ll_ag_P_fc_const_cont) > 0 else float('nan')
ll_ag_mape_fc_dyn_cont = mean_absolute_percentage_error(prices_fc, ll_ag_P_fc_dyn_cont) if n_forecast > 0 and len(ll_ag_P_fc_dyn_cont) > 0 else float('nan')
ll_ag_mape_fc_roll_cont = mean_absolute_percentage_error(prices_fc, ll_ag_P_fc_roll_cont) if n_forecast > 0 and len(ll_ag_P_fc_roll_cont) > 0 else float('nan')

print(f'Log-Linear Adjusted Gap (Mean Gap={ll_ag_gap_mean:.4f})')
print(f'  Constant K={ll_ag_k0:.6f}')
print(f'    Continuous: In-sample={ll_ag_mape_const_cont:.2%}   Forecast={ll_ag_mape_fc_const_cont:.2%}')
print(f'  Dynamic K')
print(f'    Continuous: In-sample={ll_ag_mape_dyn_cont:.2%}   Forecast={ll_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={ll_ag_k_roll_beta:.6f}')
print(f'    Continuous: In-sample={ll_ag_mape_roll_cont:.2%}   Forecast={ll_ag_mape_fc_roll_cont:.2%}')


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Log-Linear Adjusted Gap (Mean Gap=140.9514)
  Constant K=-13.855847
    Continuous: In-sample=32.20%   Forecast=190.95%
  Dynamic K
    Continuous: In-sample=1.32%   Forecast=892.22%
  Rolling K (w=12), last β=1.455504
    Continuous: In-sample=102.72%   Forecast=809.12%


/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/var/folders/9r/gxvg871x23l_vf1dydbj6q7c0000gn/T/ipykernel_86610/1149106929.py:29: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



## 📊 Section 8 — Visualizations

Interactive Plotly charts styled to match the app's light theme.
Each model section shows: Demand fit · Supply fit · K over time · EPAM price simulation.


In [159]:
def plot_2d_fit(X_plot, y_actual, y_pred, title, x_label, color):
    fig = go.Figure()
    xs = X_plot.flatten()
    sidx = np.argsort(xs)
    fig.add_trace(go.Scatter(x=xs, y=y_actual, mode='markers', name='Actual',
        marker=dict(color=T['marker'], size=7, opacity=0.85,
                    line=dict(width=1, color=T['accent']))))
    fig.add_trace(go.Scatter(x=xs[sidx], y=np.array(y_pred)[sidx], mode='lines', name='Fitted',
        line=dict(color=color, width=2.5)))
    fig.update_layout(**get_layout(title, x_label, 'Quantity'), height=540, width=960)
    return fig

def plot_k(t_fit, k_fit, t_fc, k_fc, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_fit, y=k_fit, mode='lines', name='Historical K',
        line=dict(color=T['line_red'], width=2)))
    fig.add_trace(go.Scatter(x=t_fc, y=k_fc, mode='lines', name='Forecast K',
        line=dict(color=T['line_blue'], dash='dot', width=2)))
    fig.update_layout(**get_layout(title, 'Time Step', 'K'), height=540, width=960)
    return fig

def plot_epam(t_fit, P_actual, P_sim, t_fc, P_fc, P_actual_fc, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_fit, y=P_actual, mode='lines+markers', name='Actual Price',
        line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
    fig.add_trace(go.Scatter(x=t_fit, y=P_sim, mode='lines', name='Simulated',
        line=dict(color=T['line_pink'], dash='dash', width=2)))
    fig.add_trace(go.Scatter(x=t_fc, y=P_fc, mode='lines+markers', name='Forecast',
        line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=6)))
    if len(P_actual_fc) > 0:
        fig.add_trace(go.Scatter(x=t_fc, y=P_actual_fc, mode='lines+markers', name='Actual (Holdout)',
            line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'], symbol='circle-open')))
    fig.update_layout(**get_layout(title, 'Time Step', 'Price'), height=540, width=960)
    return fig

def plot_3d(P_vals, T_vals, z_actual, func, title, surf_color):
    p_g = np.linspace(min(P_vals), max(P_vals), 20)
    t_g = np.linspace(min(T_vals), max(T_vals), 20)
    PM, TM = np.meshgrid(p_g, t_g)
    ZM = np.array([func(p, t) for p, t in zip(PM.flatten(), TM.flatten())]).reshape(PM.shape)
    ax = dict(backgroundcolor=T['scene_bg'], gridcolor=T['scene_grid'],
              zerolinecolor=T['scene_zero'], tickfont=dict(color=T['text_muted']))
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=P_vals, y=T_vals, z=z_actual, mode='markers', name='Actual',
        marker=dict(color=T['marker'], size=5, line=dict(color=T['accent'], width=1))))
    fig.add_trace(go.Surface(x=p_g, y=t_g, z=ZM, opacity=0.75, colorscale=surf_color, showscale=False))
    fig.update_layout(scene=dict(bgcolor=T['scene_bg'],
        xaxis=dict(title='Price', title_font=dict(color=T['text_muted']), **ax),
        yaxis=dict(title='Time',  title_font=dict(color=T['text_muted']), **ax),
        zaxis=dict(title='Qty',   title_font=dict(color=T['text_muted']), **ax)),
        margin=dict(l=0,r=0,b=0,t=40), paper_bgcolor='rgba(0,0,0,0)',
        title=dict(text=title, font=dict(color=T['heading'])), height=540, width=960)
    return fig

print('Plot helpers defined.')


Plot helpers defined.


### 8a — Linear


In [160]:
fig_lin_d = plot_2d_fit(X_demand_raw, y_demand, y_pred_lin_d, 'Linear Demand Fit', 'Demand Price', T['line_blue'])
save_fig(fig_lin_d, 'Linear_Demand_Fit.png')
fig_lin_s = plot_2d_fit(X_supply_raw, y_supply, y_pred_lin_s, 'Linear Supply Fit', 'Supply Price', T['line_red'])
save_fig(fig_lin_s, 'Linear_Supply_Fit.png')

# K plot
fig_k_lin = go.Figure()
fig_k_lin.add_trace(go.Scatter(x=time_steps_fit, y=lin_k_dyn_vals, mode='lines', name='Dynamic K (fit)',
    line=dict(color=T['line_blue'], width=2)))
fig_k_lin.add_trace(go.Scatter(x=time_steps_fc, y=lin_k_dyn_fc_vals, mode='lines', name='Dynamic K (forecast)',
    line=dict(color=T['line_blue'], dash='dot', width=2)))
fig_k_lin.add_trace(go.Scatter(x=time_steps_fit, y=np.full(len(time_steps_fit), lin_k0), mode='lines',
    name=f'Constant K={lin_k0:.4f}', line=dict(color=T['line_red'], dash='dash', width=2)))
fig_k_lin.add_trace(go.Scatter(x=time_steps_fit, y=lin_k_roll_vals, mode='lines', name='Rolling K (fit)',
    line=dict(color=T['line_green'], width=2)))
fig_k_lin.add_trace(go.Scatter(x=time_steps_fc, y=lin_k_roll_fc_vals, mode='lines', name='Rolling K (forecast)',
    line=dict(color=T['line_green'], dash='dot', width=2)))
fig_k_lin.update_layout(**get_layout('Linear — Adjustment Speed K', 'Time Step', 'K'), height=540, width=960)
save_fig(fig_k_lin, 'Linear_K.png')

# EPAM — Constant K
fig_lin_epam_const = plot_epam(time_steps_fit, prices_fit, lin_P_sim_const, time_steps_fc, lin_P_fc_const,
                               prices_fc, 'Linear EPAM — Constant K')
save_fig(fig_lin_epam_const, 'Linear_EPAM_Constant_K.png')

# EPAM — Dynamic K
fig_lin_epam_dyn = plot_epam(time_steps_fit, prices_fit, lin_P_sim_dyn, time_steps_fc, lin_P_fc_dyn,
                             prices_fc, 'Linear EPAM — Dynamic K')
save_fig(fig_lin_epam_dyn, 'Linear_EPAM_Dynamic_K.png')

# EPAM — Rolling K
fig_lin_epam_roll = plot_epam(time_steps_fit, prices_fit, lin_P_sim_roll, time_steps_fc, lin_P_fc_roll,
                              prices_fc, 'Linear EPAM — Rolling K')
save_fig(fig_lin_epam_roll, 'Linear_EPAM_Rolling_K.png')


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_Demand_Fit.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_Supply_Fit.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_EPAM_Constant_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_EPAM_Dynamic_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_EPAM_Rolling_K.png


### 8b — Polynomial


In [161]:
fig_poly_d = plot_2d_fit(X_demand_raw, y_demand, y_pred_poly_d, f'Poly Demand Fit (deg={poly_degree})', 'Demand Price', T['line_blue'])
save_fig(fig_poly_d, f'Poly_Demand_Fit_deg{poly_degree}.png')
fig_poly_s = plot_2d_fit(X_supply_raw, y_supply, y_pred_poly_s, f'Poly Supply Fit (deg={poly_degree})', 'Supply Price', T['line_red'])
save_fig(fig_poly_s, f'Poly_Supply_Fit_deg{poly_degree}.png')

# K plot
fig_k_poly = go.Figure()
fig_k_poly.add_trace(go.Scatter(x=time_steps_fit, y=poly_k_dyn_vals, mode='lines', name='Dynamic K (fit)',
    line=dict(color=T['line_blue'], width=2)))
fig_k_poly.add_trace(go.Scatter(x=time_steps_fc, y=poly_k_dyn_fc_vals, mode='lines', name='Dynamic K (forecast)',
    line=dict(color=T['line_blue'], dash='dot', width=2)))
fig_k_poly.add_trace(go.Scatter(x=time_steps_fit, y=np.full(len(time_steps_fit), poly_k0), mode='lines',
    name=f'Constant K={poly_k0:.4f}', line=dict(color=T['line_red'], dash='dash', width=2)))
fig_k_poly.add_trace(go.Scatter(x=time_steps_fit, y=poly_k_roll_vals, mode='lines', name='Rolling K (fit)',
    line=dict(color=T['line_green'], width=2)))
fig_k_poly.add_trace(go.Scatter(x=time_steps_fc, y=poly_k_roll_fc_vals, mode='lines', name='Rolling K (forecast)',
    line=dict(color=T['line_green'], dash='dot', width=2)))
fig_k_poly.update_layout(**get_layout('Polynomial — Adjustment Speed K', 'Time Step', 'K'), height=540, width=960)
save_fig(fig_k_poly, 'Poly_K.png')

# EPAM — Constant K
fig_poly_epam_const = plot_epam(time_steps_fit, prices_fit, poly_P_sim_const, time_steps_fc, poly_P_fc_const,
                                prices_fc, 'Polynomial EPAM — Constant K')
save_fig(fig_poly_epam_const, 'Poly_EPAM_Constant_K.png')

# EPAM — Dynamic K
fig_poly_epam_dyn = plot_epam(time_steps_fit, prices_fit, poly_P_sim_dyn, time_steps_fc, poly_P_fc_dyn,
                              prices_fc, 'Polynomial EPAM — Dynamic K')
save_fig(fig_poly_epam_dyn, 'Poly_EPAM_Dynamic_K.png')

# EPAM — Rolling K
fig_poly_epam_roll = plot_epam(time_steps_fit, prices_fit, poly_P_sim_roll, time_steps_fc, poly_P_fc_roll,
                              prices_fc, 'Polynomial EPAM — Rolling K')
save_fig(fig_poly_epam_roll, 'Polynomial_EPAM_Rolling_K.png')


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Poly_Demand_Fit_deg3.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Poly_Supply_Fit_deg3.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Poly_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Poly_EPAM_Constant_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Poly_EPAM_Dynamic_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Polynomial_EPAM_Rolling_K.png


### 8c — Multivariate


In [162]:
y_pred_na_d = np.array([na_d_func(prices_fit[i], time_steps_fit[i]) for i in range(n_fit)])
y_pred_na_s = np.array([na_s_func(prices_fit[i], time_steps_fit[i]) for i in range(n_fit)])

fig_mv_d = plot_3d(prices_fit, time_steps_fit, y_demand, na_d_func, 'Multivariate Demand Fit', 'Blues')
save_fig(fig_mv_d, 'MV_Demand_Fit.png')
fig_mv_s = plot_3d(prices_fit, time_steps_fit, y_supply, na_s_func, 'Multivariate Supply Fit', 'Reds')
save_fig(fig_mv_s, 'MV_Supply_Fit.png')

# K plot
fig_k_na = go.Figure()
fig_k_na.add_trace(go.Scatter(x=time_steps_fit, y=na_k_dyn_vals, mode='lines', name='Dynamic K (fit)',
    line=dict(color=T['line_blue'], width=2)))
fig_k_na.add_trace(go.Scatter(x=time_steps_fc, y=na_k_dyn_fc_vals, mode='lines', name='Dynamic K (forecast)',
    line=dict(color=T['line_blue'], dash='dot', width=2)))
fig_k_na.add_trace(go.Scatter(x=time_steps_fit, y=np.full(len(time_steps_fit), na_k0), mode='lines',
    name=f'Constant K={na_k0:.4f}', line=dict(color=T['line_red'], dash='dash', width=2)))
fig_k_na.add_trace(go.Scatter(x=time_steps_fit, y=na_k_roll_vals, mode='lines', name='Rolling K (fit)',
    line=dict(color=T['line_green'], width=2)))
fig_k_na.add_trace(go.Scatter(x=time_steps_fc, y=na_k_roll_fc_vals, mode='lines', name='Rolling K (forecast)',
    line=dict(color=T['line_green'], dash='dot', width=2)))
fig_k_na.update_layout(**get_layout('Multivariate — Adjustment Speed K', 'Time Step', 'K'), height=540, width=960)
save_fig(fig_k_na, 'MV_K.png')

# EPAM — Constant K
fig_mv_epam_const = plot_epam(time_steps_fit, prices_fit, na_P_sim_const, time_steps_fc, na_P_fc_const,
                              prices_fc, 'Multivariate EPAM — Constant K')
save_fig(fig_mv_epam_const, 'MV_EPAM_Constant_K.png')

# EPAM — Dynamic K
fig_mv_epam_dyn = plot_epam(time_steps_fit, prices_fit, na_P_sim_dyn, time_steps_fc, na_P_fc_dyn,
                            prices_fc, 'Multivariate EPAM — Dynamic K')
save_fig(fig_mv_epam_dyn, 'MV_EPAM_Dynamic_K.png')

# EPAM — Rolling K
fig_mv_epam_roll = plot_epam(time_steps_fit, prices_fit, na_P_sim_roll, time_steps_fc, na_P_fc_roll,
                              prices_fc, 'Multivariate EPAM — Rolling K')
save_fig(fig_mv_epam_roll, 'Multivariate_EPAM_Rolling_K.png')


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/MV_Demand_Fit.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/MV_Supply_Fit.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/MV_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/MV_EPAM_Constant_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/MV_EPAM_Dynamic_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Multivariate_EPAM_Rolling_K.png


### 8d — Log-Linear


In [163]:
fig_ll_d = plot_2d_fit(X_demand_raw, y_demand, y_pred_loglin_d, 'Log-Linear Demand Fit', 'Demand Price', T['line_blue'])
save_fig(fig_ll_d, 'LogLinear_Demand_Fit.png')
fig_ll_s = plot_2d_fit(X_supply_raw, y_supply, y_pred_loglin_s, 'Log-Linear Supply Fit', 'Supply Price', T['line_red'])
save_fig(fig_ll_s, 'LogLinear_Supply_Fit.png')

# K plot
fig_k_ll = go.Figure()
fig_k_ll.add_trace(go.Scatter(x=time_steps_fit, y=ll_k_dyn_vals, mode='lines', name='Dynamic K (fit)',
    line=dict(color=T['line_blue'], width=2)))
fig_k_ll.add_trace(go.Scatter(x=time_steps_fc, y=ll_k_dyn_fc_vals, mode='lines', name='Dynamic K (forecast)',
    line=dict(color=T['line_blue'], dash='dot', width=2)))
fig_k_ll.add_trace(go.Scatter(x=time_steps_fit, y=np.full(len(time_steps_fit), ll_k0), mode='lines',
    name=f'Constant K={ll_k0:.4f}', line=dict(color=T['line_red'], dash='dash', width=2)))
fig_k_ll.add_trace(go.Scatter(x=time_steps_fit, y=ll_k_roll_vals, mode='lines', name='Rolling K (fit)',
    line=dict(color=T['line_green'], width=2)))
fig_k_ll.add_trace(go.Scatter(x=time_steps_fc, y=ll_k_roll_fc_vals, mode='lines', name='Rolling K (forecast)',
    line=dict(color=T['line_green'], dash='dot', width=2)))
fig_k_ll.update_layout(**get_layout('Log-Linear — Adjustment Speed K', 'Time Step', 'K'), height=540, width=960)
save_fig(fig_k_ll, 'LogLinear_K.png')

# EPAM — Constant K
fig_ll_epam_const = plot_epam(time_steps_fit, prices_fit, ll_P_sim_const, time_steps_fc, ll_P_fc_const,
                             prices_fc, 'Log-Linear EPAM — Constant K')
save_fig(fig_ll_epam_const, 'LogLinear_EPAM_Constant_K.png')

# EPAM — Dynamic K
fig_ll_epam_dyn = plot_epam(time_steps_fit, prices_fit, ll_P_sim_dyn, time_steps_fc, ll_P_fc_dyn,
                           prices_fc, 'Log-Linear EPAM — Dynamic K')
save_fig(fig_ll_epam_dyn, 'LogLinear_EPAM_Dynamic_K.png')

# EPAM — Rolling K
fig_ll_epam_roll = plot_epam(time_steps_fit, prices_fit, ll_P_sim_roll, time_steps_fc, ll_P_fc_roll,
                            prices_fc, 'Log-Linear EPAM — Rolling K')
save_fig(fig_ll_epam_roll, 'LogLinear_EPAM_Rolling_K.png')


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/LogLinear_Demand_Fit.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/LogLinear_Supply_Fit.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/LogLinear_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/LogLinear_EPAM_Constant_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/LogLinear_EPAM_Dynamic_K.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/LogLinear_EPAM_Rolling_K.png


### 8e — Adjusted Gap


In [164]:
# ── Linear — Gap Analysis ──────────────────────────────────────────
fig_gap_lin = go.Figure()
fig_gap_lin.add_trace(go.Scatter(x=time_steps_fit, y=lin_ag_gap_raw, mode='lines',
    name='Raw Gap', line=dict(color=T['line_blue'], width=1.5)))
fig_gap_lin.add_trace(go.Scatter(x=time_steps_fit, y=lin_ag_gap_adj, mode='lines',
    name='Adjusted Gap', line=dict(color=T['line_pink'], width=2)))
fig_gap_lin.add_hline(y=lin_ag_gap_mean, line_dash='dash', line_color=T['line_red'],
    annotation_text=f'Mean Gap = {lin_ag_gap_mean:.2f}')
fig_gap_lin.add_hline(y=0, line_dash='dot', line_color='grey', line_width=1)
fig_gap_lin.update_layout(**get_layout('Linear — Gap Analysis (Raw vs Adjusted)', 'Time Step', 'Gap'), height=540, width=960)
save_fig(fig_gap_lin, 'Linear_Gap_Analysis.png')

# ── Linear AG EPAM — Constant K (Continuous) ───────────────────────
fig_lin_ag_const_cont = go.Figure()
fig_lin_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_lin_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=lin_ag_P_const_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(lin_ag_P_fc_const_cont) > 0:
    fig_lin_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=lin_ag_P_fc_const_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_lin_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_lin_ag_const_cont.update_layout(**get_layout('Linear AG EPAM — Constant K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_lin_ag_const_cont, 'Linear_AG_EPAM_Const_K_Continuous.png')

# ── Linear AG EPAM — Dynamic K (Continuous) ───────────────────────
fig_lin_ag_dyn_cont = go.Figure()
fig_lin_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_lin_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=lin_ag_P_dyn_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(lin_ag_P_fc_dyn_cont) > 0:
    fig_lin_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=lin_ag_P_fc_dyn_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_lin_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_lin_ag_dyn_cont.update_layout(**get_layout('Linear AG EPAM — Dynamic K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_lin_ag_dyn_cont, 'Linear_AG_EPAM_Dyn_K_Continuous.png')

# ── Linear AG EPAM — Rolling K (Continuous) ───────────────────────
fig_lin_ag_roll_cont = go.Figure()
fig_lin_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_lin_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=lin_ag_P_roll_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(lin_ag_P_fc_roll_cont) > 0:
    fig_lin_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=lin_ag_P_fc_roll_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_lin_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_lin_ag_roll_cont.update_layout(**get_layout('Linear AG EPAM — Rolling K', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_lin_ag_roll_cont, 'Linear_AG_EPAM_Roll_K_Continuous.png')



  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_Gap_Analysis.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_AG_EPAM_Const_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_AG_EPAM_Dyn_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Linear_AG_EPAM_Roll_K_Continuous.png


In [165]:
# ── Polynomial — Gap Analysis ──────────────────────────────────────────
fig_gap_poly = go.Figure()
fig_gap_poly.add_trace(go.Scatter(x=time_steps_fit, y=poly_ag_gap_raw, mode='lines',
    name='Raw Gap', line=dict(color=T['line_blue'], width=1.5)))
fig_gap_poly.add_trace(go.Scatter(x=time_steps_fit, y=poly_ag_gap_adj, mode='lines',
    name='Adjusted Gap', line=dict(color=T['line_pink'], width=2)))
fig_gap_poly.add_hline(y=poly_ag_gap_mean, line_dash='dash', line_color=T['line_red'],
    annotation_text=f'Mean Gap = {poly_ag_gap_mean:.2f}')
fig_gap_poly.add_hline(y=0, line_dash='dot', line_color='grey', line_width=1)
fig_gap_poly.update_layout(**get_layout('Polynomial — Gap Analysis (Raw vs Adjusted)', 'Time Step', 'Gap'), height=540, width=960)
save_fig(fig_gap_poly, 'Polynomial_Gap_Analysis.png')

# ── Polynomial AG EPAM — Constant K (Continuous) ───────────────────────
fig_poly_ag_const_cont = go.Figure()
fig_poly_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_poly_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=poly_ag_P_const_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(poly_ag_P_fc_const_cont) > 0:
    fig_poly_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=poly_ag_P_fc_const_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_poly_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_poly_ag_const_cont.update_layout(**get_layout('Polynomial AG EPAM — Constant K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_poly_ag_const_cont, 'Polynomial_AG_EPAM_Const_K_Continuous.png')

# ── Polynomial AG EPAM — Dynamic K (Continuous) ───────────────────────
fig_poly_ag_dyn_cont = go.Figure()
fig_poly_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_poly_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=poly_ag_P_dyn_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(poly_ag_P_fc_dyn_cont) > 0:
    fig_poly_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=poly_ag_P_fc_dyn_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_poly_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_poly_ag_dyn_cont.update_layout(**get_layout('Polynomial AG EPAM — Dynamic K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_poly_ag_dyn_cont, 'Polynomial_AG_EPAM_Dyn_K_Continuous.png')

# ── Polynomial AG EPAM — Rolling K (Continuous) ───────────────────────
fig_poly_ag_roll_cont = go.Figure()
fig_poly_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_poly_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=poly_ag_P_roll_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(poly_ag_P_fc_roll_cont) > 0:
    fig_poly_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=poly_ag_P_fc_roll_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_poly_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_poly_ag_roll_cont.update_layout(**get_layout('Polynomial AG EPAM — Rolling K', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_poly_ag_roll_cont, 'Polynomial_AG_EPAM_Roll_K_Continuous.png')



  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Polynomial_Gap_Analysis.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Polynomial_AG_EPAM_Const_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Polynomial_AG_EPAM_Dyn_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Polynomial_AG_EPAM_Roll_K_Continuous.png


In [166]:
# ── Multivariate — Gap Analysis ──────────────────────────────────────────
fig_gap_na = go.Figure()
fig_gap_na.add_trace(go.Scatter(x=time_steps_fit, y=na_ag_gap_raw, mode='lines',
    name='Raw Gap', line=dict(color=T['line_blue'], width=1.5)))
fig_gap_na.add_trace(go.Scatter(x=time_steps_fit, y=na_ag_gap_adj, mode='lines',
    name='Adjusted Gap', line=dict(color=T['line_pink'], width=2)))
fig_gap_na.add_hline(y=na_ag_gap_mean, line_dash='dash', line_color=T['line_red'],
    annotation_text=f'Mean Gap = {na_ag_gap_mean:.2f}')
fig_gap_na.add_hline(y=0, line_dash='dot', line_color='grey', line_width=1)
fig_gap_na.update_layout(**get_layout('Multivariate — Gap Analysis (Raw vs Adjusted)', 'Time Step', 'Gap'), height=540, width=960)
save_fig(fig_gap_na, 'Multivariate_Gap_Analysis.png')

# ── Multivariate AG EPAM — Constant K (Continuous) ───────────────────────
fig_na_ag_const_cont = go.Figure()
fig_na_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_na_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=na_ag_P_const_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(na_ag_P_fc_const_cont) > 0:
    fig_na_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=na_ag_P_fc_const_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_na_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_na_ag_const_cont.update_layout(**get_layout('Multivariate AG EPAM — Constant K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_na_ag_const_cont, 'Multivariate_AG_EPAM_Const_K_Continuous.png')

# ── Multivariate AG EPAM — Dynamic K (Continuous) ───────────────────────
fig_na_ag_dyn_cont = go.Figure()
fig_na_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_na_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=na_ag_P_dyn_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(na_ag_P_fc_dyn_cont) > 0:
    fig_na_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=na_ag_P_fc_dyn_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_na_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_na_ag_dyn_cont.update_layout(**get_layout('Multivariate AG EPAM — Dynamic K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_na_ag_dyn_cont, 'Multivariate_AG_EPAM_Dyn_K_Continuous.png')

# ── Multivariate AG EPAM — Rolling K (Continuous) ───────────────────────
fig_na_ag_roll_cont = go.Figure()
fig_na_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_na_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=na_ag_P_roll_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(na_ag_P_fc_roll_cont) > 0:
    fig_na_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=na_ag_P_fc_roll_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_na_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_na_ag_roll_cont.update_layout(**get_layout('Multivariate AG EPAM — Rolling K', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_na_ag_roll_cont, 'Multivariate_AG_EPAM_Roll_K_Continuous.png')



  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Multivariate_Gap_Analysis.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Multivariate_AG_EPAM_Const_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Multivariate_AG_EPAM_Dyn_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Multivariate_AG_EPAM_Roll_K_Continuous.png


In [167]:
# ── Log-Linear — Gap Analysis ──────────────────────────────────────────
fig_gap_ll = go.Figure()
fig_gap_ll.add_trace(go.Scatter(x=time_steps_fit, y=ll_ag_gap_raw, mode='lines',
    name='Raw Gap', line=dict(color=T['line_blue'], width=1.5)))
fig_gap_ll.add_trace(go.Scatter(x=time_steps_fit, y=ll_ag_gap_adj, mode='lines',
    name='Adjusted Gap', line=dict(color=T['line_pink'], width=2)))
fig_gap_ll.add_hline(y=ll_ag_gap_mean, line_dash='dash', line_color=T['line_red'],
    annotation_text=f'Mean Gap = {ll_ag_gap_mean:.2f}')
fig_gap_ll.add_hline(y=0, line_dash='dot', line_color='grey', line_width=1)
fig_gap_ll.update_layout(**get_layout('Log-Linear — Gap Analysis (Raw vs Adjusted)', 'Time Step', 'Gap'), height=540, width=960)
save_fig(fig_gap_ll, 'Log-Linear_Gap_Analysis.png')

# ── Log-Linear AG EPAM — Constant K (Continuous) ───────────────────────
fig_ll_ag_const_cont = go.Figure()
fig_ll_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_ll_ag_const_cont.add_trace(go.Scatter(x=time_steps_fit, y=ll_ag_P_const_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(ll_ag_P_fc_const_cont) > 0:
    fig_ll_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=ll_ag_P_fc_const_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_ll_ag_const_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_ll_ag_const_cont.update_layout(**get_layout('Log-Linear AG EPAM — Constant K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_ll_ag_const_cont, 'Log-Linear_AG_EPAM_Const_K_Continuous.png')

# ── Log-Linear AG EPAM — Dynamic K (Continuous) ───────────────────────
fig_ll_ag_dyn_cont = go.Figure()
fig_ll_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_ll_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fit, y=ll_ag_P_dyn_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(ll_ag_P_fc_dyn_cont) > 0:
    fig_ll_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=ll_ag_P_fc_dyn_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_ll_ag_dyn_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_ll_ag_dyn_cont.update_layout(**get_layout('Log-Linear AG EPAM — Dynamic K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_ll_ag_dyn_cont, 'Log-Linear_AG_EPAM_Dyn_K_Continuous.png')

# ── Log-Linear AG EPAM — Rolling K (Continuous) ───────────────────────
fig_ll_ag_roll_cont = go.Figure()
fig_ll_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=prices_fit, mode='lines+markers',
    name='Actual Price', line=dict(color=T['marker'], width=2), marker=dict(size=5, color=T['marker'])))
fig_ll_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fit, y=ll_ag_P_roll_cont, mode='lines',
    name='Continuous', line=dict(color=T['line_green'], dash='dash', width=2)))
if n_forecast > 0 and len(ll_ag_P_fc_roll_cont) > 0:
    fig_ll_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=ll_ag_P_fc_roll_cont, mode='lines+markers',
        name='FC Continuous', line=dict(color=T['line_green'], dash='dot', width=2), marker=dict(size=5)))
if n_forecast > 0 and len(prices_fc) > 0:
    fig_ll_ag_roll_cont.add_trace(go.Scatter(x=time_steps_fc, y=prices_fc, mode='lines+markers',
        name='Actual (Holdout)', line=dict(color=T['marker'], width=2),
        marker=dict(size=5, color=T['marker'], symbol='circle-open')))
fig_ll_ag_roll_cont.update_layout(**get_layout('Log-Linear AG EPAM — Rolling K (Continuous)', 'Time Step', 'Price'), height=540, width=960)
save_fig(fig_ll_ag_roll_cont, 'Log-Linear_AG_EPAM_Roll_K_Continuous.png')



  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Log-Linear_Gap_Analysis.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Log-Linear_AG_EPAM_Const_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Log-Linear_AG_EPAM_Dyn_K_Continuous.png


  ✅ Saved → Simulation_Result/AMPLOP 90 PPS PPL/Log-Linear_AG_EPAM_Roll_K_Continuous.png


## 📋 Section 9 — Summary

Equations, K values, and MAPE for all models (both Constant and Dynamic K).


In [168]:
def fmtv(v, prec=4):
    """Format a value: scientific notation if |v| < 0.001, else fixed-point."""
    if v == 0:
        return f"{v:.{prec}f}"
    if abs(v) < 0.001:
        exp = int(__import__('math').floor(__import__('math').log10(abs(v))))
        mant = v / (10 ** exp)
        return f"{mant:.2f} * 10^{exp}"
    return f"{v:.{prec}f}"

def fmt(c, prec=4):
    """Signed coefficient formatter with scientific notation for tiny values."""
    sign = '+' if c >= 0 else '-'
    return f"{sign}{fmtv(abs(c), prec)}"

def fmtk(s, ic):
    if abs(s) > 1e-10:
        return f"{fmt(s, 6)}*t {fmt(ic, 6)}"
    return fmtv(ic, 6)

# ── Regression equations ──────────────────────────────────────────────────
eq_lin_d  = f"Q_d = {fmtv(lin_model_d.intercept_)} {fmt(lin_model_d.coef_[0])}*p"
eq_lin_s  = f"Q_s = {fmtv(lin_model_s.intercept_)} {fmt(lin_model_s.coef_[0])}*p"
eq_poly_d = f"Q_d = {fmtv(poly_model_d.intercept_)} " + " ".join([f"{fmt(c)}*p^{i}" for i,c in enumerate(poly_model_d.coef_[1:],1)])
eq_poly_s = f"Q_s = {fmtv(poly_model_s.intercept_)} " + " ".join([f"{fmt(c)}*p^{i}" for i,c in enumerate(poly_model_s.coef_[1:],1)])
eq_na_d   = f"Q_d = {fmtv(na_d_int)} {fmt(na_d_coefs[0])}*p {fmt(na_d_coefs[1])}*t"
eq_na_s   = f"Q_s = {fmtv(na_s_int)} {fmt(na_s_coefs[0])}*p {fmt(na_s_coefs[1])}*t"

# ── D(P) - S(P) simplified ────────────────────────────────────────────────
# Linear: (a_d - a_s) + (b_d - b_s)*P
lin_c0 = lin_model_d.intercept_ - lin_model_s.intercept_
lin_c1 = lin_model_d.coef_[0]  - lin_model_s.coef_[0]
eq_lin_gap = f"D-S = {fmtv(lin_c0)} {fmt(lin_c1)}*p"

# Polynomial: term-by-term subtraction (coef_[0] is intercept, [1:] are P^1,P^2,...)
poly_c0   = poly_model_d.intercept_ - poly_model_s.intercept_
poly_gaps = poly_model_d.coef_[1:] - poly_model_s.coef_[1:]
eq_poly_gap = f"D-S = {fmtv(poly_c0)} " + " ".join([f"{fmt(c)}*p^{i}" for i,c in enumerate(poly_gaps,1)])

# Multivariate: (a_d-a_s) + (bP_d-bP_s)*P + (bt_d-bt_s)*t
na_c0  = na_d_int        - na_s_int
na_cP  = na_d_coefs[0]   - na_s_coefs[0]
na_ct  = na_d_coefs[1]   - na_s_coefs[1]
eq_na_gap = f"D-S = {fmtv(na_c0)} {fmt(na_cP)}*p {fmt(na_ct)}*t"

# ── Print summary ─────────────────────────────────────────────────────────
import sys, io
_std_io = io.StringIO()
_old_stdout = sys.stdout
sys.stdout = _std_io
print('LINEAR')
print(f'  Demand        : {eq_lin_d}   MAPE={mape_lin_d:.2%}')
print(f'  Supply        : {eq_lin_s}   MAPE={mape_lin_s:.2%}')
print(f'  D(P) - S(P)   : {eq_lin_gap}')
print(f'  Constant K={fmtv(lin_k0, 6)}')
print(f'    EPAM : dP/dt = {fmtv(lin_k0, 6)} * ({eq_lin_gap})')
print(f'    In-sample={lin_mape_sim_const:.2%}   Forecast={lin_mape_fc_const:.2%}')
print(f'  Dynamic K(t) = {fmtk(lin_k_slope, lin_k_intercept)}')
print(f'    EPAM : dP/dt = K(t) * ({eq_lin_gap})')
print(f'    In-sample={lin_mape_sim_dyn:.2%}   Forecast={lin_mape_fc_dyn:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(lin_k_roll_last_beta, 6)}')
print(f'    EPAM : dP/dt = K_roll(t) * ({eq_lin_gap})')
print(f'    In-sample={lin_mape_sim_roll:.2%}   Forecast={lin_mape_fc_roll:.2%}')
print()
print('POLYNOMIAL')
print(f'  Demand        : {eq_poly_d}   MAPE={mape_poly_d:.2%}')
print(f'  Supply        : {eq_poly_s}   MAPE={mape_poly_s:.2%}')
print(f'  D(P) - S(P)   : {eq_poly_gap}')
print(f'  Constant K={fmtv(poly_k0, 6)}')
print(f'    EPAM : dP/dt = {fmtv(poly_k0, 6)} * ({eq_poly_gap})')
print(f'    In-sample={poly_mape_sim_const:.2%}   Forecast={poly_mape_fc_const:.2%}')
print(f'  Dynamic K(t) = {fmtk(poly_k_slope, poly_k_intercept)}')
print(f'    EPAM : dP/dt = K(t) * ({eq_poly_gap})')
print(f'    In-sample={poly_mape_sim_dyn:.2%}   Forecast={poly_mape_fc_dyn:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(poly_k_roll_last_beta, 6)}')
print(f'    EPAM : dP/dt = K_roll(t) * ({eq_poly_gap})')
print(f'    In-sample={poly_mape_sim_roll:.2%}   Forecast={poly_mape_fc_roll:.2%}')
print()
print('MULTIVARIATE')
print(f'  Demand        : {eq_na_d}   MAPE={mape_d_na:.2%}')
print(f'  Supply        : {eq_na_s}   MAPE={mape_s_na:.2%}')
print(f'  D(P,t) - S(P,t) : {eq_na_gap}')
print(f'  Constant K={fmtv(na_k0, 6)}')
print(f'    EPAM : dP/dt = {fmtv(na_k0, 6)} * ({eq_na_gap})')
print(f'    In-sample={na_mape_sim_const:.2%}   Forecast={na_mape_fc_const:.2%}')
print(f'  Dynamic K(t) = {fmtk(na_k_slope, na_k_intercept)}')
print(f'    EPAM : dP/dt = K(t) * ({eq_na_gap})')
print(f'    In-sample={na_mape_sim_dyn:.2%}   Forecast={na_mape_fc_dyn:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(na_k_roll_last_beta, 6)}')
print(f'    EPAM : dP/dt = K_roll(t) * ({eq_na_gap})')
print(f'    In-sample={na_mape_sim_roll:.2%}   Forecast={na_mape_fc_roll:.2%}')
print()
print('LOG-LINEAR')
eq_ll_d = f'ln(Q_d) = {fmtv(loglin_model_d.intercept_)} {fmt(loglin_model_d.coef_[0], 6)}*P'
eq_ll_s = f'ln(Q_s) = {fmtv(loglin_model_s.intercept_)} {fmt(loglin_model_s.coef_[0], 6)}*P'
print(f'  Demand        : {eq_ll_d}   MAPE={mape_loglin_d:.2%}')
print(f'  Supply        : {eq_ll_s}   MAPE={mape_loglin_s:.2%}')
print(f'  D(P) - S(P)   : exp({fmtv(loglin_model_d.intercept_)}{fmt(loglin_model_d.coef_[0],6)}*P) - exp({fmtv(loglin_model_s.intercept_)}{fmt(loglin_model_s.coef_[0],6)}*P)')
print(f'  Constant K={fmtv(ll_k0, 6)}')
print(f'    EPAM : dP/dt = {fmtv(ll_k0, 6)} * (D(P) - S(P))')
print(f'    In-sample={ll_mape_sim_const:.2%}   Forecast={ll_mape_fc_const:.2%}')
print(f'  Dynamic K(t) = {fmtk(ll_k_slope, ll_k_intercept)}')
print(f'    EPAM : dP/dt = K(t) * (D(P) - S(P))')
print(f'    In-sample={ll_mape_sim_dyn:.2%}   Forecast={ll_mape_fc_dyn:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(ll_k_roll_last_beta, 6)}')
print(f'    EPAM : dP/dt = K_roll(t) * (D(P) - S(P))')
print(f'    In-sample={ll_mape_sim_roll:.2%}   Forecast={ll_mape_fc_roll:.2%}')

sys.stdout = _old_stdout
std_summary_text = _std_io.getvalue()
print(std_summary_text, end='')


LINEAR
  Demand        : Q_d = 518.8750 +0.0617*p   MAPE=3057.75%
  Supply        : Q_s = 6012.6712 -0.3291*p   MAPE=26.93%
  D(P) - S(P)   : D-S = -5493.7962 +0.3908*p
  Constant K=-0.064661
    EPAM : dP/dt = -0.064661 * (D-S = -5493.7962 +0.3908*p)
    In-sample=2.46%   Forecast=17.09%
  Dynamic K(t) = +0.006642*t +0.044879
    EPAM : dP/dt = K(t) * (D-S = -5493.7962 +0.3908*p)
    In-sample=2.17%   Forecast=5.57%
  Rolling K (w=12), last β=0.010406
    EPAM : dP/dt = K_roll(t) * (D-S = -5493.7962 +0.3908*p)
    In-sample=7.96%   Forecast=30.96%

POLYNOMIAL
  Demand        : Q_d = -17787401.8870 +3467.8833*p^1 -0.2253*p^2 +4.88 * 10^-6*p^3   MAPE=3197.13%
  Supply        : Q_s = 845248.7939 -212.5139*p^1 +0.0176*p^2 -4.81 * 10^-7*p^3   MAPE=27.19%
  D(P) - S(P)   : D-S = -18632650.6809 +3680.3972*p^1 -0.2429*p^2 +5.36 * 10^-6*p^3
  Constant K=-0.003556
    EPAM : dP/dt = -0.003556 * (D-S = -18632650.6809 +3680.3972*p^1 -0.2429*p^2 +5.36 * 10^-6*p^3)
    In-sample=2.35%   Forecast=16

In [169]:
# ── Adjusted Gap Summary ──────────────────────────────────────────────
import sys, io
_ag_io = io.StringIO()
_old_stdout = sys.stdout
sys.stdout = _ag_io
print('\n' + '='*70)
print('ADJUSTED GAP EPAM RESULTS')
print('='*70)
print()
print('LINEAR — Adjusted Gap')
print(f'  Mean Gap (structural bias) = {fmtv(lin_ag_gap_mean)}')
print(f'  Constant K = {fmtv(lin_ag_k0, 6)}')
print(f'    Continuous: In-sample={lin_ag_mape_const_cont:.2%}   Forecast={lin_ag_mape_fc_const_cont:.2%}')
try:    _s, _i = np.polyfit(time_steps_fit, np.nan_to_num(lin_ag_k_dyn), 1)
except: _s, _i = 0.0, float(lin_ag_k_dyn[-1])
print(f'  Dynamic K(t) = {fmtk(_s, _i)}')
print(f'    Continuous: In-sample={lin_ag_mape_dyn_cont:.2%}   Forecast={lin_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(lin_ag_k_roll_beta, 6)}')
print(f'    Continuous: In-sample={lin_ag_mape_roll_cont:.2%}   Forecast={lin_ag_mape_fc_roll_cont:.2%}')
print()
print('POLYNOMIAL — Adjusted Gap')
print(f'  Mean Gap (structural bias) = {fmtv(poly_ag_gap_mean)}')
print(f'  Constant K = {fmtv(poly_ag_k0, 6)}')
print(f'    Continuous: In-sample={poly_ag_mape_const_cont:.2%}   Forecast={poly_ag_mape_fc_const_cont:.2%}')
try:    _s, _i = np.polyfit(time_steps_fit, np.nan_to_num(poly_ag_k_dyn), 1)
except: _s, _i = 0.0, float(poly_ag_k_dyn[-1])
print(f'  Dynamic K(t) = {fmtk(_s, _i)}')
print(f'    Continuous: In-sample={poly_ag_mape_dyn_cont:.2%}   Forecast={poly_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(poly_ag_k_roll_beta, 6)}')
print(f'    Continuous: In-sample={poly_ag_mape_roll_cont:.2%}   Forecast={poly_ag_mape_fc_roll_cont:.2%}')
print()
print('MULTIVARIATE — Adjusted Gap')
print(f'  Mean Gap (structural bias) = {fmtv(na_ag_gap_mean)}')
print(f'  Constant K = {fmtv(na_ag_k0, 6)}')
print(f'    Continuous: In-sample={na_ag_mape_const_cont:.2%}   Forecast={na_ag_mape_fc_const_cont:.2%}')
try:    _s, _i = np.polyfit(time_steps_fit, np.nan_to_num(na_ag_k_dyn), 1)
except: _s, _i = 0.0, float(na_ag_k_dyn[-1])
print(f'  Dynamic K(t) = {fmtk(_s, _i)}')
print(f'    Continuous: In-sample={na_ag_mape_dyn_cont:.2%}   Forecast={na_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(na_ag_k_roll_beta, 6)}')
print(f'    Continuous: In-sample={na_ag_mape_roll_cont:.2%}   Forecast={na_ag_mape_fc_roll_cont:.2%}')
print()
print('LOG-LINEAR — Adjusted Gap')
print(f'  Mean Gap (structural bias) = {fmtv(ll_ag_gap_mean)}')
print(f'  Constant K = {fmtv(ll_ag_k0, 6)}')
print(f'    Continuous: In-sample={ll_ag_mape_const_cont:.2%}   Forecast={ll_ag_mape_fc_const_cont:.2%}')
try:    _s, _i = np.polyfit(time_steps_fit, np.nan_to_num(ll_ag_k_dyn), 1)
except: _s, _i = 0.0, float(ll_ag_k_dyn[-1])
print(f'  Dynamic K(t) = {fmtk(_s, _i)}')
print(f'    Continuous: In-sample={ll_ag_mape_dyn_cont:.2%}   Forecast={ll_ag_mape_fc_dyn_cont:.2%}')
print(f'  Rolling K (w={rolling_k_window}), last β={fmtv(ll_ag_k_roll_beta, 6)}')
print(f'    Continuous: In-sample={ll_ag_mape_roll_cont:.2%}   Forecast={ll_ag_mape_fc_roll_cont:.2%}')

sys.stdout = _old_stdout
ag_summary_text = _ag_io.getvalue()
print(ag_summary_text, end='')



ADJUSTED GAP EPAM RESULTS

LINEAR — Adjusted Gap
  Mean Gap (structural bias) = 487.3964
  Constant K = 0.000000
    Continuous: In-sample=1.50%   Forecast=13.41%
  Dynamic K(t) = -0.260505*t +4.355968
    Continuous: In-sample=1.44%   Forecast=87.36%
  Rolling K (w=12), last β=0.014863
    Continuous: In-sample=15.21%   Forecast=18.05%

POLYNOMIAL — Adjusted Gap
  Mean Gap (structural bias) = 9294.6800
  Constant K = 0.072984
    Continuous: In-sample=8.25%   Forecast=213.98%
  Dynamic K(t) = -0.025961*t +0.345252
    Continuous: In-sample=1.54%   Forecast=1348.22%
  Rolling K (w=12), last β=-0.006391
    Continuous: In-sample=36.88%   Forecast=833.07%

MULTIVARIATE — Adjusted Gap
  Mean Gap (structural bias) = 1176.8400
  Constant K = 0.210804
    Continuous: In-sample=1.78%   Forecast=9.41%
  Dynamic K(t) = -0.006042*t +0.457157
    Continuous: In-sample=1.32%   Forecast=5.83%
  Rolling K (w=12), last β=0.018791
    Continuous: In-sample=6.55%   Forecast=11.87%

LOG-LINEAR — Adjust

## 💾 Export Metrics

Automatically exports the captured summaries above into `metrics.md` within the `Simulation_Result/<item_name>` directory.

In [170]:
import datetime
import os, pathlib

OUT_DIR = pathlib.Path('Simulation_Result') / item_name
os.makedirs(OUT_DIR, exist_ok=True)
md_path = OUT_DIR / 'metrics.md'

with open(md_path, 'w', encoding='utf-8') as f:
    f.write(f"# Simulation Metrics — {item_name}\n\n")
    f.write(f"_Generated on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_\n\n")
    f.write(f"**Config**: n_fit={n_fit}, n_forecast={n_forecast}, poly_degree={poly_degree}")
    rk = CONFIG.get('rolling_k_window', 12)
    f.write(f", rolling_window={rk}\n\n")
    f.write("---\n\n")
    
    f.write("## Standard EPAM Summary\n\n")
    f.write("```\n")
    f.write(std_summary_text.rstrip() + "\n")
    f.write("```\n\n")
    
    f.write("---\n\n")
    f.write("## Adjusted Gap EPAM Summary\n\n")
    f.write("```\n")
    f.write(ag_summary_text.rstrip() + "\n")
    f.write("```\n")

print(f"✅ Metrics exported to: {md_path}")


✅ Metrics exported to: Simulation_Result/AMPLOP 90 PPS PPL/metrics.md
